In [1]:
import sys, os, json
from pathlib import Path
import pandas as pd

# --- Add repo root so `from config import ALL_DATA_JSON` works ---
def add_repo_root(markers=("config.py", ".env")):
    here = Path.cwd()
    for p in [here, *here.parents]:
        if all((p / m).exists() for m in markers):
            sys.path.insert(0, str(p))
            return p
    raise RuntimeError("Could not find repo root with config.py and .env")

repo_root = add_repo_root()

from config import ALL_DATA_JSON, SURVEY_AGGREGATED_JSON  # now this should import

import json, re
from pathlib import Path
import re, json

# Load
with open(ALL_DATA_JSON, "r") as f:
    all_data = json.load(f)
with open(SURVEY_AGGREGATED_JSON, "r") as f:
    survey_agg = json.load(f)

def norm_image_id(s):
    # "Ba2 96_1 Dy30 C1" -> "BA2_96_1_Dy30_C1"
    return "_".join(s.replace("Ba", "BA").split())

def day_from_main_id(main_id):
    m = re.search(r"(Dy\d+)", main_id, flags=re.I)
    return m.group(1).title() if m else None  # "Dy30"

agg_keys_lower = [k.lower() for k in survey_agg.keys()]

mismatched = []          # (main_id, expected_substring)
missing_in_agg = []      # image_id (normalized)
non_dy30_in_all = []     # main_ids that aren't Dy30 but have surveys
non_dy30_evals = []      # (main_id, eval_dayID, image_id) if eval not Dy30

for main_id, info in all_data.items():
    d = day_from_main_id(main_id)
    if not info.get("survey") or not info["survey"].get("evaluations"):
        continue

    # Only analyze entries whose main_id is Dy30; flag others that still have surveys
    if d != "Dy30":
        non_dy30_in_all.append(main_id)
        continue

    for ev in info["survey"]["evaluations"]:
        img_raw = ev.get("image_id", "")
        img_norm = norm_image_id(img_raw)
        ev_day = (ev.get("dayID") or "").title()

        # If any eval isn’t Dy30, flag it (surveys should be Dy30 only)
        if ev_day and ev_day != "Dy30":
            non_dy30_evals.append((main_id, ev_day, img_raw))

        # Expect substring match main_id contains image_id (+ _split{idx} if present)
        split_idx = ev.get("split_index")
        expected = img_norm + (f"_split{split_idx}" if split_idx else "")
        if expected.lower() not in main_id.lower():
            mismatched.append((main_id, expected))

        # Ensure this image exists in aggregated survey keys
        if img_norm.lower() not in agg_keys_lower:
            missing_in_agg.append(img_norm)

# Deduplicate for cleaner output
mismatched = list(dict.fromkeys(mismatched))
missing_in_agg = sorted(set(missing_in_agg))
non_dy30_in_all = sorted(set(non_dy30_in_all))
non_dy30_evals = list(dict.fromkeys(non_dy30_evals))

print("=== Dy30 survey mapping check ===")
print(f"Mismatched main_id vs expected (Dy30 only): {len(mismatched)}")
for tup in mismatched[:15]:
    print("  main_id:", tup[0], " expected_substring:", tup[1])

print(f"\nImage IDs missing from survey_aggregated keys: {len(missing_in_agg)}")
for s in missing_in_agg[:15]:
    print(" ", s)

print(f"\nEntries with surveys where main_id day != Dy30: {len(non_dy30_in_all)}")
for s in non_dy30_in_all[:10]:
    print(" ", s)

print(f"\nEvaluations not labeled Dy30 inside Dy30 main_ids: {len(non_dy30_evals)}")
for tup in non_dy30_evals[:15]:
    print("  main_id:", tup[0], " eval_dayID:", tup[1], " image_id:", tup[2])


=== Dy30 survey mapping check ===
Mismatched main_id vs expected (Dy30 only): 369
  main_id: BA1 96_1 Dy30 A1  expected_substring: BA1_96_1_Dy30_A1
  main_id: BA1 96_1 Dy30 A10  expected_substring: BA1_96_1_Dy30_A10
  main_id: BA1 96_1 Dy30 A11  expected_substring: BA1_96_1_Dy30_A11
  main_id: BA1 96_1 Dy30 A12  expected_substring: BA1_96_1_Dy30_A12
  main_id: BA1 96_1 Dy30 A2  expected_substring: BA1_96_1_Dy30_A2
  main_id: BA1 96_1 Dy30 A3  expected_substring: BA1_96_1_Dy30_A3
  main_id: BA1 96_1 Dy30 A4  expected_substring: BA1_96_1_Dy30_A4
  main_id: BA1 96_1 Dy30 A5  expected_substring: BA1_96_1_Dy30_A5
  main_id: BA1 96_1 Dy30 A6  expected_substring: BA1_96_1_Dy30_A6
  main_id: BA1 96_1 Dy30 A7  expected_substring: BA1_96_1_Dy30_A7
  main_id: BA1 96_1 Dy30 A8  expected_substring: BA1_96_1_Dy30_A8
  main_id: BA1 96_1 Dy30 A9  expected_substring: BA1_96_1_Dy30_A9
  main_id: BA1 96_1 Dy30 B1  expected_substring: BA1_96_1_Dy30_B1
  main_id: BA1 96_1 Dy30 B10  expected_substring: BA1_

In [ ]:
import sys, os, json
from pathlib import Path
import pandas as pd

# --- Add repo root so `from config import ALL_DATA_JSON` works ---
def add_repo_root(markers=("config.py", ".env")):
    here = Path.cwd()
    for p in [here, *here.parents]:
        if all((p / m).exists() for m in markers):
            sys.path.insert(0, str(p))
            return p
    raise RuntimeError("Could not find repo root with config.py and .env")

repo_root = add_repo_root()

from config import ALL_DATA_JSON, SURVEY_AGGREGATED_JSON  # now this should import

# --- Load all_data.json ---
with open(ALL_DATA_JSON, "r") as f:
    all_data = json.load(f)

print(f"Loaded {len(all_data):,} organoids")

# --- Extract splits with survey data ---
split_entries = {
    k: v for k, v in all_data.items()
    if v.get("Classification") == "Split" and "survey" in v
}

from collections import Counter

def summarize_split_ratings(entries: dict) -> pd.DataFrame:
    rows = []
    for oid, entry in entries.items():
        evals = [e["evaluation"] for e in entry.get("survey", {}).get("evaluations", [])]
        if not evals:
            continue
        c = Counter(evals)
        total = sum(c.values())
        maj_label, maj_n = c.most_common(1)[0]
        rows.append({
            "organoid_id": oid,
            "n_raters": total,
            "n_accept": c.get("Acceptable", 0),
            "n_not": c.get("Not Acceptable", 0),
            "majority_label": maj_label,
            "agreement_ratio": maj_n / total if total else None,
        })
    return pd.DataFrame(rows)

df_splits = summarize_split_ratings(split_entries)
print(df_splits.head())

# --- Optionally map to parent id (before "_split") for later comparisons ---
def parent_id(oid: str) -> str:
    return oid.split("_split")[0] if "_split" in oid else oid

df_splits["parent_id"] = df_splits["organoid_id"].apply(parent_id)


Loaded 5,168 organoids
                 organoid_id  n_raters  n_accept  n_not  majority_label  \
0  BA2 96_1 Dy28 D10 split_1         5         0      5  Not Acceptable   
1  BA2 96_1 Dy28 D10 split_2         5         0      5  Not Acceptable   
2   BA2 96_1 Dy28 E7 split_1         5         4      1      Acceptable   
3   BA2 96_1 Dy28 E7 split_2         5         4      1      Acceptable   
4   BA2 96_2 Dy28 D7 split_1         5         3      2      Acceptable   

   agreement_ratio  
0              1.0  
1              1.0  
2              0.8  
3              0.8  
4              0.6  


In [2]:
import json
import pandas as pd
from collections import defaultdict
import numpy as np
import re

# Load JSON file with NaN handling
def load_organoid_data(filepath):
    """Load JSON file, handling NaN values"""
    with open(filepath, 'r') as f:
        # Replace NaN with null for proper JSON parsing
        content = f.read().replace('NaN', 'null')
        data = json.loads(content)
    return data

def analyze_splits(data):
    """Analyze organoid splits and inter-rater agreement"""
    
    # Filter for Dy30 organoids with survey data
    dy30_organoids = {}
    for key, org in data.items():
        if (org.get('dayID') == 'Dy30' and 
            org.get('survey') and 
            org['survey'].get('evaluations') and 
            len(org['survey']['evaluations']) > 0):
            dy30_organoids[key] = org
    
    print(f"Found {len(dy30_organoids)} Dy30 organoids with survey data")
    
    # Group by parent well
    well_groups = defaultdict(list)
    
    for key, org in dy30_organoids.items():
        main_id = org.get('main_id', key)
        
        # Extract parent well ID and split index
        # Examples: "BA2_96_2_Dy30_B3_split1_nostitch" or "BA2_96_2_Dy30_B3_split2_nostitch"
        split_match = re.search(r'_split(\d+)_', main_id)
        if split_match:
            split_index = int(split_match.group(1))
            parent_well = main_id[:split_match.start()]
        elif '_nosplit' in main_id:
            parent_well = main_id[:main_id.rfind('_nosplit')]
            split_index = 0
        elif '_stitched' in main_id:
            parent_well = main_id[:main_id.rfind('_stitched')]
            split_index = -1
        else:
            parent_well = main_id
            split_index = 0
        
        # Calculate consensus for this organoid
        evaluations = org['survey']['evaluations']
        acceptable = sum(1 for e in evaluations if e.get('evaluation') == 'Acceptable')
        not_acceptable = sum(1 for e in evaluations if e.get('evaluation') == 'Not Acceptable')
        total = len(evaluations)
        
        well_groups[parent_well].append({
            'main_id': main_id,
            'split_index': split_index,
            'evaluations': evaluations,
            'acceptable': acceptable,
            'not_acceptable': not_acceptable,
            'total': total,
            'acceptable_rate': acceptable / total if total > 0 else 0,
            'consensus': 'Acceptable' if acceptable > not_acceptable else 'Not Acceptable',
            'agreement_strength': abs(acceptable - not_acceptable) / total if total > 0 else 0
        })
    
    # Find wells with multiple splits
    split_wells = {well: splits for well, splits in well_groups.items() if len(splits) > 1}
    print(f"Found {len(split_wells)} wells with multiple splits")
    
    # Analyze evil twins
    evil_twins = []
    concordant_twins = []
    ambiguous_twins = []
    
    for well, splits in split_wells.items():
        splits.sort(key=lambda x: x['split_index'])
        for i in range(len(splits)):
            for j in range(i + 1, len(splits)):
                s1, s2 = splits[i], splits[j]
                twin_pair = {
                    'well': well,
                    'split1_id': s1['main_id'],
                    'split2_id': s2['main_id'],
                    'split1_index': s1['split_index'],
                    'split2_index': s2['split_index'],
                    'split1_acceptable': s1['acceptable'],
                    'split1_not_acceptable': s1['not_acceptable'],
                    'split2_acceptable': s2['acceptable'],
                    'split2_not_acceptable': s2['not_acceptable'],
                    'split1_total': s1['total'],
                    'split2_total': s2['total'],
                    'split1_acceptable_rate': s1['acceptable_rate'],
                    'split2_acceptable_rate': s2['acceptable_rate'],
                    'split1_strength': s1['agreement_strength'],
                    'split2_strength': s2['agreement_strength'],
                }

                # --- classify relationship ---
                if s1['consensus'] != s2['consensus']:
                    twin_pair['relationship'] = 'Evil Twins'  # opposite consensus (e.g. 2/5 vs 4/5)
                    evil_twins.append(twin_pair)
                elif s1['agreement_strength'] > 0.5 and s2['agreement_strength'] > 0.5:
                    twin_pair['relationship'] = 'Concordant'
                    concordant_twins.append(twin_pair)
                else:
                    twin_pair['relationship'] = 'Ambiguous'
                    ambiguous_twins.append(twin_pair)

    
    # Calculate overall statistics
    all_split_organoids = [org for org in dy30_organoids.values() 
                          if '_split' in org.get('main_id', '')]
    
    total_splits = len(all_split_organoids)
    avg_raters = np.mean([len(org['survey']['evaluations']) 
                          for org in all_split_organoids]) if all_split_organoids else 0
    
    # Inter-rater agreement
    agreements = []
    for org in all_split_organoids:
        evals = org['survey']['evaluations']
        acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
        total = len(evals)
        agreement = max(acceptable, total - acceptable) / total if total > 0 else 0
        agreements.append(agreement)
    
    avg_agreement = np.mean(agreements) if agreements else 0
    
    # Create summary dataframe
    all_splits_df = []
    for org in all_split_organoids:
        evals = org['survey']['evaluations']
        acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
        not_acceptable = sum(1 for e in evals if e.get('evaluation') == 'Not Acceptable')
        all_splits_df.append({
            'main_id': org['main_id'],
            'acceptable': acceptable,
            'not_acceptable': not_acceptable,
            'total': len(evals),
            'acceptable_rate': acceptable / len(evals) if len(evals) > 0 else 0
        })
    
    results = {
        'summary': {
            'total_dy30': len(dy30_organoids),
            'total_splits': total_splits,
            'wells_with_multiple_splits': len(split_wells),
            'avg_raters_per_split': avg_raters,
            'avg_inter_rater_agreement': avg_agreement * 100,
            'evil_twins_count': len(evil_twins),
            'concordant_twins_count': len(concordant_twins),
            'ambiguous_twins_count': len(ambiguous_twins)
        },
        'evil_twins': evil_twins,
        'concordant_twins': concordant_twins,
        'ambiguous_twins': ambiguous_twins,
        'all_splits': all_splits_df
    }
    
    return results

def save_results(results, output_dir='output'):
    """Save analysis results to CSV files"""
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    # Save summary
    summary_df = pd.DataFrame([results['summary']])
    summary_df.to_csv(f'{output_dir}/summary_stats.csv', index=False)
    print(f"\nSummary saved to {output_dir}/summary_stats.csv")
    
    # Save evil twins
    if results['evil_twins']:
        evil_df = pd.DataFrame(results['evil_twins'])
        evil_df.to_csv(f'{output_dir}/evil_twins.csv', index=False)
        print(f"Evil twins saved to {output_dir}/evil_twins.csv ({len(results['evil_twins'])} pairs)")
    
    # Save concordant twins
    if results['concordant_twins']:
        concordant_df = pd.DataFrame(results['concordant_twins'])
        concordant_df.to_csv(f'{output_dir}/concordant_twins.csv', index=False)
        print(f"Concordant twins saved to {output_dir}/concordant_twins.csv ({len(results['concordant_twins'])} pairs)")
    
    # Save ambiguous twins
    if results['ambiguous_twins']:
        ambiguous_df = pd.DataFrame(results['ambiguous_twins'])
        ambiguous_df.to_csv(f'{output_dir}/ambiguous_twins.csv', index=False)
        print(f"Ambiguous twins saved to {output_dir}/ambiguous_twins.csv ({len(results['ambiguous_twins'])} pairs)")
    
    # Save all splits
    if results['all_splits']:
        all_splits_df = pd.DataFrame(results['all_splits'])
        all_splits_df.to_csv(f'{output_dir}/all_splits_ratings.csv', index=False)
        print(f"All splits saved to {output_dir}/all_splits_ratings.csv ({len(results['all_splits'])} organoids)")

def plot_twin_analysis(results, output_dir='output'):
    """Create visualizations of twin relationships"""
    import matplotlib.pyplot as plt
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Organoid Split Twin Analysis', fontsize=16, fontweight='bold')
    
    # 1. Twin relationship breakdown (pie chart)
    ax = axes[0, 0]
    summary = results['summary']
    sizes = [summary['concordant_twins_count'], 
             summary['ambiguous_twins_count'], 
             summary['evil_twins_count']]
    labels = ['Concordant\n(same rating, strong)', 
              'Ambiguous\n(same rating, weak)', 
              'Evil Twins\n(opposite rating)']
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    
    explode = (0.05, 0.05, 0.1) if summary['evil_twins_count'] > 0 else (0.05, 0.05, 0)
    ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', 
           startangle=90, explode=explode, textprops={'fontsize': 10})
    ax.set_title('Twin Relationship Distribution', fontsize=12, fontweight='bold')
    
    # 2. Overall rating distribution for all splits
    ax = axes[0, 1]
    all_splits_df = pd.DataFrame(results['all_splits'])
    if len(all_splits_df) > 0:
        acceptable_counts = all_splits_df['acceptable'].values
        not_acceptable_counts = all_splits_df['not_acceptable'].values
        
        x = np.arange(len(all_splits_df))
        width = 0.35
        
        ax.barh(x, acceptable_counts, width, label='Acceptable', color='green', alpha=0.7)
        ax.barh(x, -not_acceptable_counts, width, label='Not Acceptable', color='red', alpha=0.7)
        
        ax.set_xlabel('Number of Raters', fontsize=11)
        ax.set_ylabel('Split Organoid Index', fontsize=11)
        ax.set_title('Rating Distribution\nAll Split Organoids', fontsize=12, fontweight='bold')
        ax.axvline(0, color='black', linewidth=0.8)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='x')
    
    # 3. Acceptable rate histogram
    ax = axes[0, 2]
    if len(all_splits_df) > 0:
        ax.hist(all_splits_df['acceptable_rate'], bins=10, color='purple', alpha=0.7, edgecolor='black')
        ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='50% threshold')
        ax.set_xlabel('Acceptable Rate', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('Distribution of Acceptable Rates', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
    
    # 4. Concordant twins - rating strength comparison
    ax = axes[1, 0]
    if results['concordant_twins']:
        concordant_df = pd.DataFrame(results['concordant_twins'])
        
        split1_rates = concordant_df['split1_acceptable_rate'].values
        split2_rates = concordant_df['split2_acceptable_rate'].values
        
        ax.scatter(split1_rates, split2_rates, s=100, alpha=0.6, color='green', edgecolors='black')
        ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect correlation')
        
        ax.set_xlabel('Split 1 Acceptable Rate', fontsize=11)
        ax.set_ylabel('Split 2 Acceptable Rate', fontsize=11)
        ax.set_title(f'Concordant Twins (n={len(concordant_df)})\nRating Correlation', 
                    fontsize=12, fontweight='bold')
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No Concordant Twins', ha='center', va='center', fontsize=12)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
    
    # 5. Agreement strength distribution
    ax = axes[1, 1]
    if results['concordant_twins']:
        concordant_df = pd.DataFrame(results['concordant_twins'])
        
        split1_strength = concordant_df['split1_strength'].values
        split2_strength = concordant_df['split2_strength'].values
        
        ax.hist(split1_strength, bins=10, alpha=0.5, label='Split 1', color='blue')
        ax.hist(split2_strength, bins=10, alpha=0.5, label='Split 2', color='orange')
        
        ax.set_xlabel('Agreement Strength', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('Rater Agreement Strength\nConcordant Twins', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
    else:
        ax.text(0.5, 0.5, 'No Concordant Twins', ha='center', va='center', fontsize=12)
    
    # 6. Inter-rater agreement summary
    ax = axes[1, 2]
    summary_data = {
        'Metric': ['Avg Raters\nper Split', 'Avg Agreement\n(%)', 'Wells with\nMultiple Splits'],
        'Value': [summary['avg_raters_per_split'], 
                 summary['avg_inter_rater_agreement'],
                 summary['wells_with_multiple_splits']]
    }
    
    colors_bar = ['#3498db', '#2ecc71', '#9b59b6']
    ax.bar(summary_data['Metric'], summary_data['Value'], color=colors_bar, alpha=0.7, edgecolor='black')
    ax.set_ylabel('Value', fontsize=11)
    ax.set_title('Summary Statistics', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, (metric, val) in enumerate(zip(summary_data['Metric'], summary_data['Value'])):
        ax.text(i, val + max(summary_data['Value'])*0.02, f'{val:.1f}', 
               ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/twin_analysis_summary.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"\n✓ Visualization saved to {output_dir}/twin_analysis_summary.png")

def print_summary(results):
    """Print summary statistics"""
    summary = results['summary']
    
    print("\n" + "="*60)
    print("ORGANOID SPLIT ANALYSIS SUMMARY")
    print("="*60)
    print(f"\nTotal Dy30 organoids with surveys: {summary['total_dy30']}")
    print(f"Total split organoids: {summary['total_splits']}")
    print(f"Wells with multiple splits: {summary['wells_with_multiple_splits']}")
    print(f"Average raters per split: {summary['avg_raters_per_split']:.1f}")
    print(f"Average inter-rater agreement: {summary['avg_inter_rater_agreement']:.1f}%")
    
    print("\n" + "-"*60)
    print("TWIN RELATIONSHIP BREAKDOWN")
    print("-"*60)
    print(f"Evil Twins (opposite consensus): {summary['evil_twins_count']}")
    print(f"Concordant Twins (same consensus, strong): {summary['concordant_twins_count']}")
    print(f"Ambiguous Twins (same consensus, weak): {summary['ambiguous_twins_count']}")
    
    if results['evil_twins']:
        print("\n" + "-"*60)
        print("EVIL TWIN EXAMPLES (first 10)")
        print("-"*60)
        for i, twin in enumerate(results['evil_twins'][:10], 1):
            print(f"\n{i}. Well: {twin['well']}")
            print(f"   Split {twin['split1_index']}: {twin['split1_consensus']} "
                  f"({twin['split1_acceptable_rate']*100:.0f}% acceptable)")
            print(f"   Split {twin['split2_index']}: {twin['split2_consensus']} "
                  f"({twin['split2_acceptable_rate']*100:.0f}% acceptable)")
            print(f"   Raters: {twin['rater_count']}")
    


# Main execution
if __name__ == "__main__":
    # CHANGE THIS to your JSON file path
    input_file = ALL_DATA_JSON  # <-- PUT YOUR FILE PATH HERE
    
    print(f"Loading data from {input_file}...")
    data = load_organoid_data(input_file)
    print(f"Loaded {len(data)} total organoid entries")
    
    print("\nAnalyzing splits...")
    results = analyze_splits(data)
    
    print_summary(results)
    
    print("\nSaving results...")
    save_results(results, output_dir='organoid_analysis_output')
    
    print("\nGenerating visualizations...")
    plot_twin_analysis(results, output_dir='organoid_analysis_output')
    
    print("\n✓ Analysis complete!")

Loading data from /net/projects2/promega/data-analysis/output/all_data.json...
Loaded 5168 total organoid entries

Analyzing splits...
Found 369 Dy30 organoids with survey data
Found 13 wells with multiple splits

ORGANOID SPLIT ANALYSIS SUMMARY

Total Dy30 organoids with surveys: 369
Total split organoids: 27
Wells with multiple splits: 13
Average raters per split: 5.4
Average inter-rater agreement: 85.2%

------------------------------------------------------------
TWIN RELATIONSHIP BREAKDOWN
------------------------------------------------------------
Evil Twins (opposite consensus): 9
Concordant Twins (same consensus, strong): 3
Ambiguous Twins (same consensus, weak): 1

------------------------------------------------------------
EVIL TWIN EXAMPLES (first 10)
------------------------------------------------------------

1. Well: BA2_96_1_Dy30_C1


KeyError: 'split1_consensus'

In [6]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec
import numpy as np
import re
from pathlib import Path
from collections import defaultdict

def load_organoid_data(filepath):
    """Load JSON file, handling NaN values"""
    with open(filepath, 'r') as f:
        content = f.read().replace('NaN', 'null')
        data = json.loads(content)
    return data

def analyze_splits(data):
    """Analyze organoid splits and inter-rater agreement"""
    
    # Filter for Dy30 organoids with survey data
    dy30_organoids = {}
    for key, org in data.items():
        if (org.get('dayID') == 'Dy30' and 
            org.get('survey') and 
            org['survey'].get('evaluations') and 
            len(org['survey']['evaluations']) > 0):
            dy30_organoids[key] = org
    
    print(f"Found {len(dy30_organoids)} Dy30 organoids with survey data")
    
    # Group by parent well
    well_groups = defaultdict(list)
    
    for key, org in dy30_organoids.items():
        main_id = org.get('main_id', key)
        
        # Extract parent well ID and split index
        split_match = re.search(r'_split(\d+)_', main_id)
        if split_match:
            split_index = int(split_match.group(1))
            parent_well = main_id[:split_match.start()]
        elif '_nosplit' in main_id:
            parent_well = main_id[:main_id.rfind('_nosplit')]
            split_index = 0
        elif '_stitched' in main_id:
            parent_well = main_id[:main_id.rfind('_stitched')]
            split_index = -1
        else:
            parent_well = main_id
            split_index = 0
        
        # Calculate consensus for this organoid
        evaluations = org['survey']['evaluations']
        acceptable = sum(1 for e in evaluations if e.get('evaluation') == 'Acceptable')
        not_acceptable = sum(1 for e in evaluations if e.get('evaluation') == 'Not Acceptable')
        total = len(evaluations)
        
        well_groups[parent_well].append({
            'main_id': main_id,
            'split_index': split_index,
            'evaluations': evaluations,
            'acceptable': acceptable,
            'not_acceptable': not_acceptable,
            'total': total,
            'acceptable_rate': acceptable / total if total > 0 else 0,
            'consensus': 'Acceptable' if acceptable > not_acceptable else 'Not Acceptable',
            'agreement_strength': abs(acceptable - not_acceptable) / total if total > 0 else 0
        })
    
    # Find wells with multiple splits
    split_wells = {well: splits for well, splits in well_groups.items() if len(splits) > 1}
    print(f"Found {len(split_wells)} wells with multiple splits")
    
    # Analyze evil twins
    evil_twins = []
    concordant_twins = []
    ambiguous_twins = []
    
    # --- classification section ---
    for well, splits in split_wells.items():
        splits.sort(key=lambda x: x['split_index'])
        for i in range(len(splits)):
            for j in range(i + 1, len(splits)):
                s1, s2 = splits[i], splits[j]
                
                twin_pair = {
                    'well': well,
                    'split1_id': s1['main_id'],
                    'split2_id': s2['main_id'],
                    'split1_index': s1['split_index'],
                    'split2_index': s2['split_index'],
                    'split1_consensus': s1['consensus'],
                    'split2_consensus': s2['consensus'],
                    'split1_acceptable': s1['acceptable'],
                    'split2_acceptable': s2['acceptable'],
                    'split1_not_acceptable': s1['not_acceptable'],
                    'split2_not_acceptable': s2['not_acceptable'],
                    'split1_strength': s1['agreement_strength'],
                    'split2_strength': s2['agreement_strength'],
                    'split1_acceptable_rate': s1['acceptable_rate'],
                    'split2_acceptable_rate': s2['acceptable_rate'],
                    'rater_count': min(s1['total'], s2['total'])  # or something else if you prefer
                }

                if s1['consensus'] != s2['consensus']:
                    twin_pair['relationship'] = 'Evil Twins'
                    evil_twins.append(twin_pair)
                elif s1['agreement_strength'] > 0.5 and s2['agreement_strength'] > 0.5:
                    twin_pair['relationship'] = 'Concordant'
                    concordant_twins.append(twin_pair)
                else:
                    twin_pair['relationship'] = 'Ambiguous'
                    ambiguous_twins.append(twin_pair)

    # Calculate overall statistics
    all_split_organoids = [
        org for org in dy30_organoids.values()
        if '_split' in org.get('main_id', '')
    ]
    
    total_splits = len(all_split_organoids)
    avg_raters = np.mean([
        len(org['survey']['evaluations'])
        for org in all_split_organoids
    ]) if all_split_organoids else 0
    
    # Inter-rater agreement
    agreements = []
    for org in all_split_organoids:
        evals = org['survey']['evaluations']
        acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
        total = len(evals)
        agreement = max(acceptable, total - acceptable) / total if total > 0 else 0
        agreements.append(agreement)
    
    avg_agreement = np.mean(agreements) if agreements else 0
    
    # Create summary dataframe
    all_splits_df = []
    for org in all_split_organoids:
        evals = org['survey']['evaluations']
        acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
        not_acceptable = sum(1 for e in evals if e.get('evaluation') == 'Not Acceptable')
        all_splits_df.append({
            'main_id': org['main_id'],
            'acceptable': acceptable,
            'not_acceptable': not_acceptable,
            'total': len(evals),
            'acceptable_rate': acceptable / len(evals) if len(evals) > 0 else 0
        })
    
    results = {
        'summary': {
            'total_dy30': len(dy30_organoids),
            'total_splits': total_splits,
            'wells_with_multiple_splits': len(split_wells),
            'avg_raters_per_split': avg_raters,
            'avg_inter_rater_agreement': avg_agreement * 100,
            'evil_twins_count': len(evil_twins),
            'concordant_twins_count': len(concordant_twins),
            'ambiguous_twins_count': len(ambiguous_twins)
        },
        'evil_twins': evil_twins,
        'concordant_twins': concordant_twins,
        'ambiguous_twins': ambiguous_twins,
        'all_splits': all_splits_df
    }
    
    return results


def get_image_path(data, main_id):
    """Get img image path for a given main_id"""
    for key, org in data.items():
        if org.get('main_id') == main_id:
            if 'processed' in org and 'img_path' in org['processed']:
                return org['processed']['img_path']
    return None

def get_rating_summary(data, main_id):
    """Get rating summary for display"""
    for key, org in data.items():
        if org.get('main_id') == main_id:
            if 'survey' in org and 'evaluations' in org['survey']:
                evals = org['survey']['evaluations']
                acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
                not_acceptable = sum(1 for e in evals if e.get('evaluation') == 'Not Acceptable')
                total = len(evals)
                return {
                    'acceptable': acceptable,
                    'not_acceptable': not_acceptable,
                    'total': total,
                    'rate': acceptable / total if total > 0 else 0,
                    'consensus': 'Acceptable' if acceptable > not_acceptable else 'Not Acceptable'
                }
    return None

def visualize_twin_pair(data, twin_pair, ax1, ax2):
    """Visualize a single twin pair"""
    
    # Get image paths
    img1_path = get_image_path(data, twin_pair['split1_id'])
    img2_path = get_image_path(data, twin_pair['split2_id'])
    
    # Get rating summaries
    rating1 = get_rating_summary(data, twin_pair['split1_id'])
    rating2 = get_rating_summary(data, twin_pair['split2_id'])
    
    # Load and display images
    if img1_path and Path(img1_path).exists():
        img1 = mpimg.imread(img1_path)
        ax1.imshow(img1)
    else:
        ax1.text(0.5, 0.5, f'Image not found\n{img1_path}', ha='center', va='center', 
                transform=ax1.transAxes, fontsize=8)
        ax1.set_facecolor('#f0f0f0')
    
    if img2_path and Path(img2_path).exists():
        img2 = mpimg.imread(img2_path)
        ax2.imshow(img2)
    else:
        ax2.text(0.5, 0.5, f'Image not found\n{img2_path}', ha='center', va='center', 
                transform=ax2.transAxes, fontsize=8)
        ax2.set_facecolor('#f0f0f0')
    
    # Format titles with ratings
    if rating1:
        color1 = 'green' if rating1['consensus'] == 'Acceptable' else 'red'
        title1 = f"Split {twin_pair['split1_index']}\n{rating1['consensus']}\n({rating1['acceptable']}/{rating1['total']} Acceptable)"
        ax1.set_title(title1, fontsize=10, fontweight='bold', color=color1)
    else:
        ax1.set_title(f"Split {twin_pair['split1_index']}", fontsize=10)
    
    if rating2:
        color2 = 'green' if rating2['consensus'] == 'Acceptable' else 'red'
        title2 = f"Split {twin_pair['split2_index']}\n{rating2['consensus']}\n({rating2['acceptable']}/{rating2['total']} Acceptable)"
        ax2.set_title(title2, fontsize=10, fontweight='bold', color=color2)
    else:
        ax2.set_title(f"Split {twin_pair['split2_index']}", fontsize=10)
    
    ax1.axis('off')
    ax2.axis('off')

def visualize_twins(data, twins_list, category_name, output_file, max_pairs=10):
    """Visualize multiple twin pairs in a grid"""
    
    n_pairs = min(len(twins_list), max_pairs)
    if n_pairs == 0:
        print(f"No {category_name} to visualize")
        return
    
    # Create figure with grid
    fig = plt.figure(figsize=(12, 4 * n_pairs))
    fig.suptitle(f'{category_name} Twin Pairs (n={len(twins_list)})', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    for i, twin in enumerate(twins_list[:n_pairs]):
        # Create subplot for each twin pair
        ax1 = plt.subplot(n_pairs, 2, i*2 + 1)
        ax2 = plt.subplot(n_pairs, 2, i*2 + 2)
        
        # Add well label on the left
        well_label = twin['well'].split('_')[-3:]
        fig.text(0.01, 1 - (i + 0.5) / n_pairs, f"Well: {' '.join(well_label)}", 
                fontsize=11, fontweight='bold', va='center', rotation=90)
        
        visualize_twin_pair(data, twin, ax1, ax2)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    print(f"Saved {category_name} visualization to {output_file}")
    plt.close()

def create_twin_montage(data, twins_list, category_name, output_file, max_pairs=20):
    """Create a compact montage of twin pairs"""
    
    n_pairs = min(len(twins_list), max_pairs)
    if n_pairs == 0:
        print(f"No {category_name} to visualize")
        return
    
    # Calculate grid dimensions
    n_cols = 4  # 2 pairs per row (4 images)
    n_rows = (n_pairs * 2 + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    fig.suptitle(f'{category_name} Twin Pairs (n={len(twins_list)})', 
                 fontsize=16, fontweight='bold')
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    # Flatten axes for easy iteration
    axes_flat = axes.flatten()
    
    img_idx = 0
    for pair_idx, twin in enumerate(twins_list[:n_pairs]):
        # Get image paths and ratings
        img1_path = get_image_path(data, twin['split1_id'])
        img2_path = get_image_path(data, twin['split2_id'])
        rating1 = get_rating_summary(data, twin['split1_id'])
        rating2 = get_rating_summary(data, twin['split2_id'])
        
        # Display split 1
        if img_idx < len(axes_flat):
            ax = axes_flat[img_idx]
            if img1_path and Path(img1_path).exists():
                img = mpimg.imread(img1_path)
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, 'Not found', ha='center', va='center')
                ax.set_facecolor('#f0f0f0')
            
            if rating1:
                color = 'green' if rating1['consensus'] == 'Acceptable' else 'red'
                title = f"S{twin['split1_index']}: {rating1['acceptable']}/{rating1['total']}"
                ax.set_title(title, fontsize=9, color=color, fontweight='bold')
            ax.axis('off')
            img_idx += 1
        
        # Display split 2
        if img_idx < len(axes_flat):
            ax = axes_flat[img_idx]
            if img2_path and Path(img2_path).exists():
                img = mpimg.imread(img2_path)
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, 'Not found', ha='center', va='center')
                ax.set_facecolor('#f0f0f0')
            
            if rating2:
                color = 'green' if rating2['consensus'] == 'Acceptable' else 'red'
                title = f"S{twin['split2_index']}: {rating2['acceptable']}/{rating2['total']}"
                ax.set_title(title, fontsize=9, color=color, fontweight='bold')
            ax.axis('off')
            img_idx += 1
    
    # Hide unused axes
    for idx in range(img_idx, len(axes_flat)):
        axes_flat[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    print(f"Saved {category_name} montage to {output_file}")
    plt.close()

def print_summary(results):
    """Print summary statistics"""
    summary = results['summary']
    
    print("\n" + "="*60)
    print("ORGANOID SPLIT ANALYSIS SUMMARY")
    print("="*60)
    print(f"\nTotal Dy30 organoids with surveys: {summary['total_dy30']}")
    print(f"Total split organoids: {summary['total_splits']}")
    print(f"Wells with multiple splits: {summary['wells_with_multiple_splits']}")
    print(f"Average raters per split: {summary['avg_raters_per_split']:.1f}")
    print(f"Average inter-rater agreement: {summary['avg_inter_rater_agreement']:.1f}%")
    
    print("\n" + "-"*60)
    print("TWIN RELATIONSHIP BREAKDOWN")
    print("-"*60)
    print(f"Evil Twins (opposite consensus): {summary['evil_twins_count']}")
    print(f"Concordant Twins (same consensus, strong): {summary['concordant_twins_count']}")
    print(f"Ambiguous Twins (same consensus, weak): {summary['ambiguous_twins_count']}")
    
    if results['evil_twins']:
        print("\n" + "-"*60)
        print("EVIL TWIN EXAMPLES (first 10)")
        print("-"*60)
        for i, twin in enumerate(results['evil_twins'][:10], 1):
            print(f"\n{i}. Well: {twin['well']}")
            print(f"   Split {twin['split1_index']}: {twin['split1_consensus']} "
                  f"({twin['split1_acceptable_rate']*100:.0f}% acceptable)")
            print(f"   Split {twin['split2_index']}: {twin['split2_consensus']} "
                  f"({twin['split2_acceptable_rate']*100:.0f}% acceptable)")
            print(f"   Raters: {twin['rater_count']}")

# Main execution
if __name__ == "__main__":
    # File path
    input_file = "/net/projects2/promega/data-analysis/output/all_data.json"
    output_dir = "split_analysis_output"
    Path(output_dir).mkdir(exist_ok=True)
    
    print(f"Loading data from {input_file}...")
    data = load_organoid_data(input_file)
    print(f"Loaded {len(data)} total organoid entries")
    
    print("\nAnalyzing splits...")
    results = analyze_splits(data)
    
    print_summary(results)
    
    # Visualize Evil Twins (if any)
    print("\n" + "="*60)
    print("GENERATING VISUALIZATIONS")
    print("="*60)
    
    if results['evil_twins']:
        print(f"\nVisualizing {len(results['evil_twins'])} Evil Twin pairs...")
        visualize_twins(data, results['evil_twins'], "Evil Twins", 
                      f"{output_dir}/evil_twins_visualization.png", max_pairs=10)
        create_twin_montage(data, results['evil_twins'], "Evil Twins",
                          f"{output_dir}/evil_twins_montage.png", max_pairs=20)
    else:
        print("\nNo Evil Twins found - skipping visualization")
    
    # Visualize Concordant Twins
    if results['concordant_twins']:
        print(f"\nVisualizing {len(results['concordant_twins'])} Concordant Twin pairs...")
        visualize_twins(data, results['concordant_twins'], "Concordant Twins",
                      f"{output_dir}/concordant_twins_visualization.png", max_pairs=10)
        create_twin_montage(data, results['concordant_twins'], "Concordant Twins",
                          f"{output_dir}/concordant_twins_montage.png", max_pairs=20)
    
    # Visualize Ambiguous Twins
    if results['ambiguous_twins']:
        print(f"\nVisualizing {len(results['ambiguous_twins'])} Ambiguous Twin pairs...")
        visualize_twins(data, results['ambiguous_twins'], "Ambiguous Twins",
                      f"{output_dir}/ambiguous_twins_visualization.png", max_pairs=10)
        create_twin_montage(data, results['ambiguous_twins'], "Ambiguous Twins",
                          f"{output_dir}/ambiguous_twins_montage.png", max_pairs=20)
    
    print("\n✓ Analysis and visualization complete!")
    print(f"Check {output_dir}/ for output images")

Loading data from /net/projects2/promega/data-analysis/output/all_data.json...
Loaded 5168 total organoid entries

Analyzing splits...
Found 369 Dy30 organoids with survey data
Found 13 wells with multiple splits

ORGANOID SPLIT ANALYSIS SUMMARY

Total Dy30 organoids with surveys: 369
Total split organoids: 27
Wells with multiple splits: 13
Average raters per split: 5.4
Average inter-rater agreement: 85.2%

------------------------------------------------------------
TWIN RELATIONSHIP BREAKDOWN
------------------------------------------------------------
Evil Twins (opposite consensus): 9
Concordant Twins (same consensus, strong): 3
Ambiguous Twins (same consensus, weak): 1

------------------------------------------------------------
EVIL TWIN EXAMPLES (first 10)
------------------------------------------------------------

1. Well: BA2_96_1_Dy30_C1
   Split 1: Acceptable (60% acceptable)
   Split 2: Not Acceptable (0% acceptable)
   Raters: 5

2. Well: BA2_96_1_Dy30_D10
   Split 1: N

In [7]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec
import numpy as np
import re
from pathlib import Path
from collections import defaultdict

def load_organoid_data(filepath):
    """Load JSON file, handling NaN values"""
    with open(filepath, 'r') as f:
        content = f.read().replace('NaN', 'null')
        data = json.loads(content)
    return data

def extract_well_info(main_id):
    """Extract BA, wellID, and day from main_id"""
    # Example: BA2_96_1_Dy30_C1_split1_nostitch
    parts = main_id.split('_')
    
    # Find day
    day_idx = [i for i, p in enumerate(parts) if p.startswith('Dy')]
    if day_idx:
        day = parts[day_idx[0]]
        # BA is everything before day
        ba = '_'.join(parts[:day_idx[0]])
        # Well is after day
        well = parts[day_idx[0] + 1]
        return ba, well, day
    return None, None, None

def normalize_ba(ba_str):
    """Normalize BA string for comparison (handle spaces/underscores)"""
    # Remove spaces and underscores, convert to lowercase for comparison
    return ba_str.replace(' ', '').replace('_', '').lower()

def track_split_history(data, dy30_splits):
    """Track each split back through time to find parent organoids"""
    
    # Available days in order
    all_days = ['Dy03', 'Dy06', 'Dy08', 'Dy10', 'Dy13', 'Dy15', 'Dy17', 'Dy21', 'Dy24', 'Dy28', 'Dy30']
    
    split_histories = []
    
    for split_main_id in dy30_splits:
        ba, well, _ = extract_well_info(split_main_id)
        if not ba or not well:
            continue
        
        # Normalize BA for matching
        ba_normalized = normalize_ba(ba)
        
        history = {
            'dy30_main_id': split_main_id,
            'ba': ba,
            'well': well,
            'timeline': {}
        }
        
        # Look through all days for this BA/well combination
        for day in all_days:
            # Search for entries matching this BA, well, and day
            for key, org in data.items():
                org_ba = org.get('BA', '')
                org_ba_normalized = normalize_ba(org_ba)
                
                if (org_ba_normalized == ba_normalized and 
                    org.get('wellID') == well and 
                    org.get('dayID') == day):
                    
                    main_id = org.get('main_id', '')
                    classification = org.get('Classification', '')
                    overlay_path = org.get('processed', {}).get('overlay_path', '')
                    
                    # Determine if split or presplit
                    if '_split' in main_id:
                        split_match = re.search(r'_split(\d+)_', main_id)
                        split_idx = int(split_match.group(1)) if split_match else 0
                        status = f'split{split_idx}'
                    else:
                        status = 'presplit'
                        split_idx = 0
                    
                    if day not in history['timeline']:
                        history['timeline'][day] = []
                    
                    history['timeline'][day].append({
                        'main_id': main_id,
                        'status': status,
                        'split_index': split_idx,
                        'classification': classification,
                        'overlay_path': overlay_path
                    })
        
        split_histories.append(history)
    
    return split_histories

def find_split_onset(timeline):
    """Find the first day when split appears"""
    all_days = ['Dy03', 'Dy06', 'Dy08', 'Dy10', 'Dy13', 'Dy15', 'Dy17', 'Dy21', 'Dy24', 'Dy28', 'Dy30']
    
    for day in all_days:
        if day in timeline:
            entries = timeline[day]
            if any(e['status'].startswith('split') for e in entries):
                return day
    return None

def visualize_split_timeline(data, split_histories, output_dir, max_wells=10):
    """Create timeline visualizations for split development"""
    
    Path(output_dir).mkdir(exist_ok=True)
    
    # Group by well (combine split1 and split2 from same well)
    wells_to_plot = {}
    for hist in split_histories:
        well_key = f"{hist['ba']}_{hist['well']}"
        if well_key not in wells_to_plot:
            wells_to_plot[well_key] = hist
    
    print(f"\nFound {len(wells_to_plot)} unique wells with splits on Dy30")
    
    # Create comprehensive timeline for each well
    for idx, (well_key, hist) in enumerate(list(wells_to_plot.items())[:max_wells]):
        
        timeline = hist['timeline']
        split_onset = find_split_onset(timeline)
        
        print(f"\n{idx+1}. {well_key}")
        print(f"   Split onset: {split_onset}")
        print(f"   Days tracked: {sorted(timeline.keys())}")
        
        # Determine which days to show
        days_to_show = []
        all_days = ['Dy03', 'Dy06', 'Dy08', 'Dy10', 'Dy13', 'Dy15', 'Dy17', 'Dy21', 'Dy24', 'Dy28', 'Dy30']
        
        # Show 2-3 presplit days before split onset, then all split days
        if split_onset:
            onset_idx = all_days.index(split_onset)
            # Get 2 days before onset (if available)
            for i in range(max(0, onset_idx - 2), onset_idx):
                if all_days[i] in timeline:
                    days_to_show.append(all_days[i])
            # Add all days from onset onwards
            for day in all_days[onset_idx:]:
                if day in timeline:
                    days_to_show.append(day)
        else:
            # No split found, just show available days
            days_to_show = [d for d in all_days if d in timeline]
        
        if not days_to_show:
            continue
        
        # Count total images to show
        total_images = sum(len(timeline[day]) for day in days_to_show)
        
        # Create figure
        fig, axes = plt.subplots(1, total_images, figsize=(4 * total_images, 5))
        if total_images == 1:
            axes = [axes]
        
        fig.suptitle(f'Split Development Timeline: {well_key.replace("_", " ")}', 
                     fontsize=14, fontweight='bold')
        
        img_idx = 0
        for day in days_to_show:
            entries = timeline[day]
            
            for entry in entries:
                if img_idx >= len(axes):
                    break
                
                ax = axes[img_idx]
                overlay_path = entry['overlay_path']
                
                # Load and display image
                if overlay_path and Path(overlay_path).exists():
                    img = mpimg.imread(overlay_path)
                    ax.imshow(img)
                else:
                    ax.text(0.5, 0.5, 'Image not found', ha='center', va='center')
                    ax.set_facecolor('#f0f0f0')
                
                # Create title
                status_str = entry['status']
                if status_str == 'presplit':
                    color = 'blue'
                    title = f"{day}\nPre-split"
                else:
                    color = 'red'
                    title = f"{day}\n{status_str.capitalize()}"
                
                # Add arrow if split onset
                if day == split_onset and entry['status'] == 'presplit':
                    title = f"{day}\nPre-split\n(last before split)"
                    color = 'orange'
                elif day == split_onset and entry['status'].startswith('split'):
                    title = f"{day}\n{status_str.capitalize()}\n(SPLIT ONSET)"
                    color = 'darkred'
                
                ax.set_title(title, fontsize=10, fontweight='bold', color=color)
                ax.axis('off')
                
                img_idx += 1
        
        # Hide unused axes
        for i in range(img_idx, len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        safe_well_name = well_key.replace(' ', '_').replace('/', '_')
        plt.savefig(f"{output_dir}/timeline_{idx+1}_{safe_well_name}.png", 
                   dpi=150, bbox_inches='tight')
        plt.close()
    
    print(f"\n✓ Timeline visualizations saved to {output_dir}/")

def create_split_onset_summary(split_histories, output_dir):
    """Create summary of when splits first appear"""
    
    onset_summary = []
    
    for hist in split_histories:
        split_onset = find_split_onset(hist['timeline'])
        
        onset_summary.append({
            'well': f"{hist['ba']}_{hist['well']}",
            'split_onset_day': split_onset if split_onset else 'Unknown',
            'days_tracked': len(hist['timeline']),
            'timeline_days': ','.join(sorted(hist['timeline'].keys()))
        })
    
    # Convert to DataFrame and save
    df = pd.DataFrame(onset_summary)
    
    # Count split onsets by day
    onset_counts = df['split_onset_day'].value_counts().sort_index()
    
    print("\n" + "="*60)
    print("SPLIT ONSET SUMMARY")
    print("="*60)
    print(f"\nTotal wells with splits: {len(df)}")
    print("\nSplit onset distribution:")
    for day, count in onset_counts.items():
        print(f"  {day}: {count} wells")
    
    # Save detailed summary
    df.to_csv(f"{output_dir}/split_onset_summary.csv", index=False)
    print(f"\n✓ Detailed summary saved to {output_dir}/split_onset_summary.csv")
    
    return df

def create_montage_comparison(data, split_histories, output_dir, max_wells=12):
    """Create compact montage comparing presplit vs split states"""
    
    wells_dict = {}
    for hist in split_histories:
        well_key = f"{hist['ba']}_{hist['well']}"
        wells_dict[well_key] = hist
    
    n_wells = min(len(wells_dict), max_wells)
    
    if n_wells == 0:
        print("No wells to visualize in montage")
        return
    
    # Create montage: each row is one well showing [presplit] → [split1] [split2]
    fig, axes = plt.subplots(n_wells, 4, figsize=(16, 4 * n_wells))
    if n_wells == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle('Pre-Split vs Split Comparison (Dy30)', fontsize=16, fontweight='bold')
    
    for idx, (well_key, hist) in enumerate(list(wells_dict.items())[:n_wells]):
        timeline = hist['timeline']
        split_onset = find_split_onset(timeline)
        
        # Find last presplit image (day before split onset)
        all_days = ['Dy03', 'Dy06', 'Dy08', 'Dy10', 'Dy13', 'Dy15', 'Dy17', 'Dy21', 'Dy24', 'Dy28', 'Dy30']
        last_presplit_img = None
        
        if split_onset:
            onset_idx = all_days.index(split_onset)
            # Look backwards for presplit
            for i in range(onset_idx - 1, -1, -1):
                day = all_days[i]
                if day in timeline:
                    presplit_entries = [e for e in timeline[day] if e['status'] == 'presplit']
                    if presplit_entries:
                        last_presplit_img = presplit_entries[0]
                        break
        
        # Get Dy30 splits
        dy30_splits = []
        if 'Dy30' in timeline:
            dy30_splits = sorted(timeline['Dy30'], key=lambda x: x['split_index'])
        
        # Plot presplit
        ax = axes[idx, 0]
        if last_presplit_img and last_presplit_img['overlay_path']:
            if Path(last_presplit_img['overlay_path']).exists():
                img = mpimg.imread(last_presplit_img['overlay_path'])
                ax.imshow(img)
                # Get day from main_id
                day_match = re.search(r'Dy\d+', last_presplit_img['main_id'])
                day_str = day_match.group(0) if day_match else 'Unknown'
                ax.set_title(f"Pre-split\n({day_str})", fontsize=10, fontweight='bold', color='blue')
            else:
                ax.text(0.5, 0.5, 'Not found', ha='center', va='center')
        else:
            ax.text(0.5, 0.5, 'No presplit\nimage', ha='center', va='center')
            ax.set_facecolor('#f0f0f0')
        ax.axis('off')
        
        # Plot arrow
        ax = axes[idx, 1]
        ax.text(0.5, 0.5, '→\nSPLIT', ha='center', va='center', 
               fontsize=16, fontweight='bold', color='red')
        ax.axis('off')
        
        # Plot splits
        for split_idx, entry in enumerate(dy30_splits[:2]):  # Max 2 splits
            ax = axes[idx, 2 + split_idx]
            if entry['overlay_path'] and Path(entry['overlay_path']).exists():
                img = mpimg.imread(entry['overlay_path'])
                ax.imshow(img)
                ax.set_title(f"Dy30 {entry['status'].capitalize()}", 
                           fontsize=10, fontweight='bold', color='darkred')
            else:
                ax.text(0.5, 0.5, 'Not found', ha='center', va='center')
                ax.set_facecolor('#f0f0f0')
            ax.axis('off')
        
        # Hide unused split column if only 1 split
        if len(dy30_splits) < 2:
            axes[idx, 3].axis('off')
        
        # Add well label
        fig.text(0.02, 1 - (idx + 0.5) / n_wells, well_key.replace('_', ' '), 
                fontsize=9, fontweight='bold', va='center', rotation=90)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/presplit_vs_split_montage.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"\n✓ Comparison montage saved to {output_dir}/presplit_vs_split_montage.png")

# Main execution
if __name__ == "__main__":
    input_file = "/net/projects2/promega/data-analysis/output/all_data.json"
    
    print("="*60)
    print("LONGITUDINAL SPLIT DEVELOPMENT TRACKER")
    print("="*60)
    
    print(f"\nLoading data from {input_file}...")
    data = load_organoid_data(input_file)
    print(f"Loaded {len(data)} organoid entries")
    
    # Find all Dy30 splits
    print("\nFinding Dy30 splits...")
    dy30_splits = []
    for key, org in data.items():
        if (org.get('dayID') == 'Dy30' and 
            '_split' in org.get('main_id', '')):
            dy30_splits.append(org['main_id'])
    
    print(f"Found {len(dy30_splits)} split organoids on Dy30")
    
    # Track split histories
    print("\nTracking split development through time...")
    split_histories = track_split_history(data, dy30_splits)
    print(f"Successfully tracked {len(split_histories)} split histories")
    
    # Create split onset summary
    onset_df = create_split_onset_summary(split_histories, output_dir)
    
    # Create visualizations
    print("\nGenerating timeline visualizations...")
    visualize_split_timeline(data, split_histories, output_dir, max_wells=10)
    
    print("\nGenerating comparison montage...")
    create_montage_comparison(data, split_histories, output_dir, max_wells=12)
    
    print("\n" + "="*60)
    print("✓ ANALYSIS COMPLETE!")
    print("="*60)
    print(f"Check '{output_dir}/' for:")
    print("  - Individual timeline visualizations (timeline_*.png)")
    print("  - Pre-split vs split montage (presplit_vs_split_montage.png)")
    print("  - Split onset summary (split_onset_summary.csv)")

LONGITUDINAL SPLIT DEVELOPMENT TRACKER

Loading data from /net/projects2/promega/data-analysis/output/all_data.json...
Loaded 5168 organoid entries

Finding Dy30 splits...
Found 28 split organoids on Dy30

Tracking split development through time...
Successfully tracked 28 split histories

SPLIT ONSET SUMMARY

Total wells with splits: 28

Split onset distribution:
  Dy17: 2 wells
  Dy21: 10 wells
  Dy24: 12 wells
  Dy28: 4 wells

✓ Detailed summary saved to split_analysis_output/split_onset_summary.csv

Generating timeline visualizations...

Found 14 unique wells with splits on Dy30

1. BA2_96_1_C1
   Split onset: Dy28
   Days tracked: ['Dy03', 'Dy06', 'Dy08', 'Dy10', 'Dy13', 'Dy15', 'Dy17', 'Dy21', 'Dy24', 'Dy28', 'Dy30']

2. BA2_96_1_D10
   Split onset: Dy24
   Days tracked: ['Dy03', 'Dy06', 'Dy08', 'Dy10', 'Dy13', 'Dy15', 'Dy17', 'Dy21', 'Dy24', 'Dy28', 'Dy30']

3. BA2_96_1_E1
   Split onset: Dy24
   Days tracked: ['Dy03', 'Dy06', 'Dy08', 'Dy10', 'Dy13', 'Dy15', 'Dy17', 'Dy21', 'Dy24

In [8]:

import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
import re
from collections import defaultdict

def load_organoid_data(filepath):
    """Load JSON file, handling NaN values"""
    with open(filepath, 'r') as f:
        content = f.read().replace('NaN', 'null')
        data = json.loads(content)
    return data

def normalize_ba(ba_str):
    """Normalize BA string for comparison"""
    return ba_str.replace(' ', '').replace('_', '').lower()

def extract_mask_area(mask_path):
    """Extract area from binary mask (count white pixels)"""
    try:
        if not Path(mask_path).exists():
            return None
        
        mask = Image.open(mask_path).convert('L')
        mask_array = np.array(mask)
        
        # Count non-zero pixels (assuming mask is binary or grayscale)
        area_pixels = np.sum(mask_array > 0)
        
        return area_pixels
    except Exception as e:
        print(f"Error reading mask {mask_path}: {e}")
        return None

def calculate_area_um2(area_pixels, um_per_px_x, um_per_px_y):
    """Convert pixel area to µm²"""
    if area_pixels is None or um_per_px_x is None or um_per_px_y is None:
        return None
    
    # Area = pixels * (µm/px_x) * (µm/px_y)
    area_um2 = area_pixels * um_per_px_x * um_per_px_y
    return area_um2

def build_size_timeline(data):
    """Build timeline of size measurements for all organoids"""
    
    all_days = ['Dy03', 'Dy06', 'Dy08', 'Dy10', 'Dy13', 'Dy15', 'Dy17', 'Dy21', 'Dy24', 'Dy28', 'Dy30']
    
    # Group by BA + wellID
    well_groups = defaultdict(lambda: defaultdict(list))
    
    for key, org in data.items():
        ba = org.get('BA', '')
        well = org.get('wellID', '')
        day = org.get('dayID', '')
        
        if not ba or not well or not day:
            continue
        
        ba_norm = normalize_ba(ba)
        well_key = f"{ba_norm}_{well}"
        
        # Get mask info
        processed = org.get('processed', {})
        mask_path = processed.get('mask_path', '')
        um_per_px_x = processed.get('final_um_per_px_x')
        um_per_px_y = processed.get('final_um_per_px_y')
        main_id = org.get('main_id', '')
        
        # Determine if split
        is_split = '_split' in main_id
        split_match = re.search(r'_split(\d+)_', main_id)
        split_index = int(split_match.group(1)) if split_match else 0
        
        # Get rating if available
        rating = None
        if 'survey' in org and 'evaluations' in org['survey']:
            evals = org['survey']['evaluations']
            if evals:
                acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
                total = len(evals)
                rating = 'Acceptable' if acceptable > total/2 else 'Not Acceptable'
        
        # Extract area
        area_pixels = extract_mask_area(mask_path)
        area_um2 = calculate_area_um2(area_pixels, um_per_px_x, um_per_px_y)
        
        well_groups[well_key][day].append({
            'main_id': main_id,
            'day': day,
            'is_split': is_split,
            'split_index': split_index,
            'mask_path': mask_path,
            'area_pixels': area_pixels,
            'area_um2': area_um2,
            'um_per_px_x': um_per_px_x,
            'um_per_px_y': um_per_px_y,
            'rating': rating,
            'ba': ba,
            'well': well
        })
    
    return well_groups, all_days

def create_growth_curves(well_groups, all_days, output_dir, max_wells=15):
    """Create growth curve plots for individual wells"""
    
    Path(output_dir).mkdir(exist_ok=True)
    
    # Filter wells that have splits on Dy30
    split_wells = {}
    nonsplit_wells = {}
    
    for well_key, timeline in well_groups.items():
        if 'Dy30' in timeline:
            has_split = any(org['is_split'] for org in timeline['Dy30'])
            if has_split:
                split_wells[well_key] = timeline
            else:
                nonsplit_wells[well_key] = timeline
    
    print(f"\nFound {len(split_wells)} wells with splits")
    print(f"Found {len(nonsplit_wells)} wells without splits")
    
    # Plot individual growth curves for split wells
    fig, axes = plt.subplots(3, 5, figsize=(20, 12))
    axes = axes.flatten()
    
    fig.suptitle('Growth Curves: Wells That Split by Dy30', fontsize=16, fontweight='bold')
    
    for idx, (well_key, timeline) in enumerate(list(split_wells.items())[:15]):
        ax = axes[idx]
        
        # Collect data for plotting
        for day_idx, day in enumerate(all_days):
            if day not in timeline:
                continue
            
            for org in timeline[day]:
                if org['area_um2'] is not None:
                    color = 'blue' if not org['is_split'] else ('red' if org['split_index'] == 1 else 'orange')
                    marker = 'o' if not org['is_split'] else 's'
                    label = 'presplit' if not org['is_split'] else f"split{org['split_index']}"
                    
                    ax.scatter(day_idx, org['area_um2'], color=color, marker=marker, s=50, alpha=0.7)
        
        ax.set_xticks(range(len(all_days)))
        ax.set_xticklabels(all_days, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Area (µm²)', fontsize=9)
        ax.set_title(well_key.replace('_', ' ')[:20], fontsize=9)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(len(list(split_wells.items())[:15]), 15):
        axes[idx].axis('off')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='blue', label='Presplit'),
        Patch(facecolor='red', label='Split 1'),
        Patch(facecolor='orange', label='Split 2')
    ]
    fig.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/growth_curves_split_wells.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Growth curves saved to {output_dir}/growth_curves_split_wells.png")

def compare_split_vs_nonsplit(well_groups, all_days, output_dir):
    """Compare size distributions: splits vs non-splits at Dy30"""
    
    dy30_splits = []
    dy30_nonsplits = []
    
    for well_key, timeline in well_groups.items():
        if 'Dy30' not in timeline:
            continue
        
        for org in timeline['Dy30']:
            if org['area_um2'] is not None:
                data_point = {
                    'well': well_key,
                    'area_um2': org['area_um2'],
                    'area_pixels': org['area_pixels'],
                    'rating': org['rating'],
                    'is_split': org['is_split'],
                    'split_index': org['split_index']
                }
                
                if org['is_split']:
                    dy30_splits.append(data_point)
                else:
                    dy30_nonsplits.append(data_point)
    
    splits_df = pd.DataFrame(dy30_splits)
    nonsplits_df = pd.DataFrame(dy30_nonsplits)
    
    print(f"\nDy30 Comparison:")
    print(f"  Splits: {len(splits_df)} organoids")
    print(f"  Non-splits: {len(nonsplits_df)} organoids")
    
    if len(splits_df) > 0:
        print(f"\n  Split mean area: {splits_df['area_um2'].mean():.0f} µm²")
        print(f"  Split median area: {splits_df['area_um2'].median():.0f} µm²")
    
    if len(nonsplits_df) > 0:
        print(f"  Non-split mean area: {nonsplits_df['area_um2'].mean():.0f} µm²")
        print(f"  Non-split median area: {nonsplits_df['area_um2'].median():.0f} µm²")
    
    # Create comparison plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Size distribution
    ax = axes[0]
    if len(nonsplits_df) > 0:
        ax.hist(nonsplits_df['area_um2'], bins=30, alpha=0.5, label='Non-split', color='blue')
    if len(splits_df) > 0:
        ax.hist(splits_df['area_um2'], bins=30, alpha=0.5, label='Split', color='red')
    ax.set_xlabel('Area (µm²)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Size Distribution at Dy30', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Box plot
    ax = axes[1]
    data_to_plot = []
    labels_to_plot = []
    if len(nonsplits_df) > 0:
        data_to_plot.append(nonsplits_df['area_um2'])
        labels_to_plot.append('Non-split')
    if len(splits_df) > 0:
        data_to_plot.append(splits_df['area_um2'])
        labels_to_plot.append('Split')
    
    if data_to_plot:
        bp = ax.boxplot(data_to_plot, labels=labels_to_plot, patch_artist=True)
        bp['boxes'][0].set_facecolor('blue')
        if len(bp['boxes']) > 1:
            bp['boxes'][1].set_facecolor('red')
    ax.set_ylabel('Area (µm²)', fontsize=12)
    ax.set_title('Size Comparison', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Rating comparison for splits
    ax = axes[2]
    if len(splits_df) > 0:
        rated_splits = splits_df[splits_df['rating'].notna()]
        if len(rated_splits) > 0:
            acceptable = rated_splits[rated_splits['rating'] == 'Acceptable']
            not_acceptable = rated_splits[rated_splits['rating'] == 'Not Acceptable']
            
            ax.bar(['Acceptable', 'Not Acceptable'], 
                   [len(acceptable), len(not_acceptable)],
                   color=['green', 'red'], alpha=0.6)
            ax.set_ylabel('Count', fontsize=12)
            ax.set_title('Split Ratings at Dy30', fontsize=12, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            
            # Add size info
            if len(acceptable) > 0:
                ax.text(0, len(acceptable) + 1, 
                       f'Mean: {acceptable["area_um2"].mean():.0f} µm²',
                       ha='center', fontsize=9)
            if len(not_acceptable) > 0:
                ax.text(1, len(not_acceptable) + 1,
                       f'Mean: {not_acceptable["area_um2"].mean():.0f} µm²',
                       ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/dy30_split_comparison.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Comparison plot saved to {output_dir}/dy30_split_comparison.png")
    
    # Save data
    if len(splits_df) > 0:
        splits_df.to_csv(f"{output_dir}/dy30_splits_sizes.csv", index=False)
    if len(nonsplits_df) > 0:
        nonsplits_df.to_csv(f"{output_dir}/dy30_nonsplits_sizes.csv", index=False)

def analyze_growth_rates(well_groups, all_days, output_dir):
    """Calculate and compare growth rates"""
    
    growth_data = []
    
    for well_key, timeline in well_groups.items():
        # Get chronological measurements
        measurements = []
        for day in all_days:
            if day not in timeline:
                continue
            for org in timeline[day]:
                if org['area_um2'] is not None:
                    measurements.append({
                        'day': day,
                        'day_num': int(day.replace('Dy', '')),
                        'area_um2': org['area_um2'],
                        'is_split': org['is_split'],
                        'split_index': org['split_index'],
                        'rating': org['rating']
                    })
        
        if len(measurements) < 2:
            continue
        
        # Sort by day
        measurements.sort(key=lambda x: x['day_num'])
        
        # Calculate growth rate (final - initial) / days
        initial = measurements[0]
        final = measurements[-1]
        
        days_elapsed = final['day_num'] - initial['day_num']
        if days_elapsed > 0:
            growth_rate = (final['area_um2'] - initial['area_um2']) / days_elapsed
            
            growth_data.append({
                'well': well_key,
                'initial_area': initial['area_um2'],
                'final_area': final['area_um2'],
                'days_elapsed': days_elapsed,
                'growth_rate_um2_per_day': growth_rate,
                'has_split_at_end': final['is_split'],
                'final_rating': final['rating'],
                'n_timepoints': len(measurements)
            })
    
    growth_df = pd.DataFrame(growth_data)
    
    if len(growth_df) > 0:
        print(f"\nGrowth Rate Analysis:")
        print(f"  Total tracked: {len(growth_df)} wells")
        
        split_growth = growth_df[growth_df['has_split_at_end'] == True]
        nonsplit_growth = growth_df[growth_df['has_split_at_end'] == False]
        
        print(f"\n  Wells that split: {len(split_growth)}")
        if len(split_growth) > 0:
            print(f"    Mean growth rate: {split_growth['growth_rate_um2_per_day'].mean():.1f} µm²/day")
            print(f"    Median growth rate: {split_growth['growth_rate_um2_per_day'].median():.1f} µm²/day")
        
        print(f"\n  Wells that didn't split: {len(nonsplit_growth)}")
        if len(nonsplit_growth) > 0:
            print(f"    Mean growth rate: {nonsplit_growth['growth_rate_um2_per_day'].mean():.1f} µm²/day")
            print(f"    Median growth rate: {nonsplit_growth['growth_rate_um2_per_day'].median():.1f} µm²/day")
        
        growth_df.to_csv(f"{output_dir}/growth_rates.csv", index=False)
        print(f"\n✓ Growth rates saved to {output_dir}/growth_rates.csv")
        
        # Plot growth rates (normalized)
        fig, ax = plt.subplots(figsize=(10, 6))
        
        if len(split_growth) > 0:
            ax.hist(split_growth['growth_rate_um2_per_day'], bins=20, 
                   alpha=0.5, label=f'Split (n={len(split_growth)})', 
                   color='red', density=True)
        if len(nonsplit_growth) > 0:
            ax.hist(nonsplit_growth['growth_rate_um2_per_day'], bins=20,
                   alpha=0.5, label=f'Non-split (n={len(nonsplit_growth)})', 
                   color='blue', density=True)
        
        ax.set_xlabel('Growth Rate (µm²/day)', fontsize=12)
        ax.set_ylabel('Density (normalized)', fontsize=12)
        ax.set_title('Growth Rate Distribution (Normalized)', fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{output_dir}/growth_rate_distribution.png", dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"✓ Growth rate plot saved to {output_dir}/growth_rate_distribution.png")

# Main execution
if __name__ == "__main__":
    input_file = "/net/projects2/promega/data-analysis/output/all_data.json"
    
    print("="*60)
    print("ORGANOID SIZE & GROWTH ANALYSIS")
    print("="*60)
    
    print(f"\nLoading data from {input_file}...")
    data = load_organoid_data(input_file)
    print(f"Loaded {len(data)} organoid entries")
    
    print("\nBuilding size timelines...")
    well_groups, all_days = build_size_timeline(data)
    print(f"Tracked {len(well_groups)} unique wells")
    
    print("\nGenerating growth curves...")
    create_growth_curves(well_groups, all_days, output_dir, max_wells=15)
    
    print("\nComparing splits vs non-splits at Dy30...")
    compare_split_vs_nonsplit(well_groups, all_days, output_dir)
    
    print("\nAnalyzing growth rates...")
    analyze_growth_rates(well_groups, all_days, output_dir)
    
    print("\n" + "="*60)
    print("✓ ANALYSIS COMPLETE!")
    print("="*60)
    print(f"Check '{output_dir}/' for:")
    print("  - growth_curves_split_wells.png")
    print("  - dy30_split_comparison.png")
    print("  - growth_rate_distribution.png")
    print("  - CSVs with detailed size/growth data")

ORGANOID SIZE & GROWTH ANALYSIS

Loading data from /net/projects2/promega/data-analysis/output/all_data.json...
Loaded 5168 organoid entries

Building size timelines...
Tracked 475 unique wells

Generating growth curves...

Found 14 wells with splits
Found 442 wells without splits
✓ Growth curves saved to split_analysis_output/growth_curves_split_wells.png

Comparing splits vs non-splits at Dy30...

Dy30 Comparison:
  Splits: 28 organoids
  Non-splits: 442 organoids

  Split mean area: 1225537 µm²
  Split median area: 1054112 µm²
  Non-split mean area: 1618392 µm²
  Non-split median area: 1331530 µm²


/tmp/ipykernel_563607/696374226.py:243: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=labels_to_plot, patch_artist=True)


✓ Comparison plot saved to split_analysis_output/dy30_split_comparison.png

Analyzing growth rates...

Growth Rate Analysis:
  Total tracked: 475 wells

  Wells that split: 14
    Mean growth rate: 40874.8 µm²/day
    Median growth rate: 30384.8 µm²/day

  Wells that didn't split: 461
    Mean growth rate: 48816.1 µm²/day
    Median growth rate: 39700.2 µm²/day

✓ Growth rates saved to split_analysis_output/growth_rates.csv
✓ Growth rate plot saved to split_analysis_output/growth_rate_distribution.png

✓ ANALYSIS COMPLETE!
Check 'split_analysis_output/' for:
  - growth_curves_split_wells.png
  - dy30_split_comparison.png
  - growth_rate_distribution.png
  - CSVs with detailed size/growth data


In [10]:
import json
import pandas as pd
from collections import defaultdict
import numpy as np
import re

# Load JSON file with NaN handling
def load_organoid_data(filepath):
    """Load JSON file, handling NaN values"""
    with open(filepath, 'r') as f:
        # Replace NaN with null for proper JSON parsing
        content = f.read().replace('NaN', 'null')
        data = json.loads(content)
    return data

def get_organoid_area(org):
    """Extract area from processed metadata, return None if unavailable."""
    processed = org.get('processed', {})
    # Adjust these keys to match your schema
    if 'area' in processed:
        return processed['area']
    if 'area_pixels' in processed:
        return processed['area_pixels']
    return None

def build_area_lookup(data):
    """Return dict: main_id -> area (or None)."""
    area_lookup = {}
    for key, org in data.items():
        main_id = org.get('main_id', key)
        area_lookup[main_id] = get_organoid_area(org)
    return area_lookup

def plot_area_based_charts(data, results, output_dir='output'):
    import matplotlib.pyplot as plt
    import os
    os.makedirs(output_dir, exist_ok=True)

    area_lookup = build_area_lookup(data)

    # 1. Area vs acceptable_rate for all splits
    all_splits_df = pd.DataFrame(results['all_splits'])
    all_splits_df['area'] = all_splits_df['main_id'].map(area_lookup)
    all_splits_df = all_splits_df.dropna(subset=['area'])

    if len(all_splits_df) > 0:
        plt.figure(figsize=(7, 5))
        plt.scatter(all_splits_df['area'], all_splits_df['acceptable_rate'],
                    alpha=0.7, edgecolors='black')
        plt.xlabel('Organoid Area', fontsize=12)
        plt.ylabel('Acceptable Rate', fontsize=12)
        plt.title('Organoid Area vs Rating', fontsize=13, fontweight='bold')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{output_dir}/area_vs_rating.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"Saved area vs rating plot to {output_dir}/area_vs_rating.png")
    else:
        print("No area info available for area vs rating plot")

    # 2. Twin asymmetry: area difference vs rating difference, colored by relationship
    twin_rows = []
    for rel_name, twin_list in [
        ('Concordant', results['concordant_twins']),
        ('Ambiguous', results['ambiguous_twins']),
        ('Evil Twins', results['evil_twins'])
    ]:
        for twin in twin_list:
            a1 = area_lookup.get(twin['split1_id'])
            a2 = area_lookup.get(twin['split2_id'])
            if a1 is None or a2 is None:
                continue

            # Symmetric measure: normalized difference and rating difference
            rating_diff = twin['split2_acceptable_rate'] - twin['split1_acceptable_rate']
            area_diff_norm = (a2 - a1) / ((a1 + a2) / 2.0) if (a1 + a2) > 0 else 0.0

            twin_rows.append({
                'relationship': rel_name,
                'rating_diff': rating_diff,
                'area_diff_norm': area_diff_norm
            })

    if len(twin_rows) > 0:
        df_twins = pd.DataFrame(twin_rows)

        plt.figure(figsize=(7, 5))
        for rel_name, color, marker in [
            ('Concordant', '#2ecc71', 'o'),
            ('Ambiguous', '#f39c12', 's'),
            ('Evil Twins', '#e74c3c', 'X')
        ]:
            sub = df_twins[df_twins['relationship'] == rel_name]
            if len(sub) == 0:
                continue
            plt.scatter(sub['area_diff_norm'], sub['rating_diff'],
                        alpha=0.8, edgecolors='black', s=80,
                        label=f'{rel_name} (n={len(sub)})', marker=marker, linewidth=1.3)

        plt.axhline(0, color='gray', linestyle='--', alpha=0.7)
        plt.axvline(0, color='gray', linestyle='--', alpha=0.7)

        plt.xlabel('Normalized Area Difference\n((A2 - A1) / mean(A1, A2))', fontsize=11)
        plt.ylabel('Rating Difference\n(split2 - split1 acceptable rate)', fontsize=11)
        plt.title('Twin Asymmetry: Size vs Rating', fontsize=13, fontweight='bold')
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=9)
        plt.tight_layout()
        plt.savefig(f'{output_dir}/twin_area_rating_asymmetry.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"Saved twin area–rating asymmetry plot to {output_dir}/twin_area_rating_asymmetry.png")
    else:
        print("No twin pairs with area info for asymmetry plot")

def analyze_splits(data):
    """Analyze organoid splits and inter-rater agreement"""
    
    # Filter for Dy30 organoids with survey data
    dy30_organoids = {}
    for key, org in data.items():
        if (org.get('dayID') == 'Dy30' and 
            org.get('survey') and 
            org['survey'].get('evaluations') and 
            len(org['survey']['evaluations']) > 0):
            dy30_organoids[key] = org
    
    print(f"Found {len(dy30_organoids)} Dy30 organoids with survey data")
    
    # Group by parent well
    well_groups = defaultdict(list)
    
    for key, org in dy30_organoids.items():
        main_id = org.get('main_id', key)
        
        # Extract parent well ID and split index
        # Examples: "BA2_96_2_Dy30_B3_split1_nostitch" or "BA2_96_2_Dy30_B3_split2_nostitch"
        split_match = re.search(r'_split(\d+)_', main_id)
        if split_match:
            split_index = int(split_match.group(1))
            parent_well = main_id[:split_match.start()]
        elif '_nosplit' in main_id:
            parent_well = main_id[:main_id.rfind('_nosplit')]
            split_index = 0
        elif '_stitched' in main_id:
            parent_well = main_id[:main_id.rfind('_stitched')]
            split_index = -1
        else:
            parent_well = main_id
            split_index = 0
        
        # Calculate consensus for this organoid
        evaluations = org['survey']['evaluations']
        acceptable = sum(1 for e in evaluations if e.get('evaluation') == 'Acceptable')
        not_acceptable = sum(1 for e in evaluations if e.get('evaluation') == 'Not Acceptable')
        total = len(evaluations)
        
        well_groups[parent_well].append({
            'main_id': main_id,
            'split_index': split_index,
            'evaluations': evaluations,
            'acceptable': acceptable,
            'not_acceptable': not_acceptable,
            'total': total,
            'acceptable_rate': acceptable / total if total > 0 else 0,
            'consensus': 'Acceptable' if acceptable > not_acceptable else 'Not Acceptable',
            'agreement_strength': abs(acceptable - not_acceptable) / total if total > 0 else 0
        })
    
    # Find wells with multiple splits
    split_wells = {well: splits for well, splits in well_groups.items() if len(splits) > 1}
    print(f"Found {len(split_wells)} wells with multiple splits")
    
    # Analyze evil twins
    evil_twins = []
    concordant_twins = []
    ambiguous_twins = []
    
    for well, splits in split_wells.items():
        # Sort by split index
        splits.sort(key=lambda x: x['split_index'])
        
        # Compare all pairs
        for i in range(len(splits)):
            for j in range(i + 1, len(splits)):
                split1 = splits[i]
                split2 = splits[j]
                
                twin_pair = {
                    'well': well,
                    'split1_id': split1['main_id'],
                    'split2_id': split2['main_id'],
                    'split1_index': split1['split_index'],
                    'split2_index': split2['split_index'],
                    'split1_consensus': split1['consensus'],
                    'split2_consensus': split2['consensus'],
                    'split1_acceptable_rate': split1['acceptable_rate'],
                    'split2_acceptable_rate': split2['acceptable_rate'],
                    'split1_strength': split1['agreement_strength'],
                    'split2_strength': split2['agreement_strength'],
                    'rater_count': split1['total']
                }
                
                # Classify twin relationship
                if split1['consensus'] != split2['consensus']:
                    # Evil twins - opposite consensus
                    twin_pair['relationship'] = 'Evil Twins'
                    evil_twins.append(twin_pair)
                elif split1['agreement_strength'] > 0.5 and split2['agreement_strength'] > 0.5:
                    # Strong concordance
                    twin_pair['relationship'] = 'Concordant'
                    concordant_twins.append(twin_pair)
                else:
                    # Weak agreement
                    twin_pair['relationship'] = 'Ambiguous'
                    ambiguous_twins.append(twin_pair)
    
    # Calculate overall statistics
    all_split_organoids = [org for org in dy30_organoids.values() 
                          if '_split' in org.get('main_id', '')]
    
    total_splits = len(all_split_organoids)
    avg_raters = np.mean([len(org['survey']['evaluations']) 
                          for org in all_split_organoids]) if all_split_organoids else 0
    
    # Inter-rater agreement
    agreements = []
    for org in all_split_organoids:
        evals = org['survey']['evaluations']
        acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
        total = len(evals)
        agreement = max(acceptable, total - acceptable) / total if total > 0 else 0
        agreements.append(agreement)
    
    avg_agreement = np.mean(agreements) if agreements else 0
    
    # Create summary dataframe
    all_splits_df = []
    for org in all_split_organoids:
        evals = org['survey']['evaluations']
        acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
        not_acceptable = sum(1 for e in evals if e.get('evaluation') == 'Not Acceptable')
        all_splits_df.append({
            'main_id': org['main_id'],
            'acceptable': acceptable,
            'not_acceptable': not_acceptable,
            'total': len(evals),
            'acceptable_rate': acceptable / len(evals) if len(evals) > 0 else 0
        })
    
    results = {
        'summary': {
            'total_dy30': len(dy30_organoids),
            'total_splits': total_splits,
            'wells_with_multiple_splits': len(split_wells),
            'avg_raters_per_split': avg_raters,
            'avg_inter_rater_agreement': avg_agreement * 100,
            'evil_twins_count': len(evil_twins),
            'concordant_twins_count': len(concordant_twins),
            'ambiguous_twins_count': len(ambiguous_twins)
        },
        'evil_twins': evil_twins,
        'concordant_twins': concordant_twins,
        'ambiguous_twins': ambiguous_twins,
        'all_splits': all_splits_df
    }
    
    return results

def save_results(results, output_dir='output'):
    """Save analysis results to CSV files"""
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    # Save summary
    summary_df = pd.DataFrame([results['summary']])
    summary_df.to_csv(f'{output_dir}/summary_stats.csv', index=False)
    print(f"\nSummary saved to {output_dir}/summary_stats.csv")
    
    # Save evil twins
    if results['evil_twins']:
        evil_df = pd.DataFrame(results['evil_twins'])
        evil_df.to_csv(f'{output_dir}/evil_twins.csv', index=False)
        print(f"Evil twins saved to {output_dir}/evil_twins.csv ({len(results['evil_twins'])} pairs)")
    
    # Save concordant twins
    if results['concordant_twins']:
        concordant_df = pd.DataFrame(results['concordant_twins'])
        concordant_df.to_csv(f'{output_dir}/concordant_twins.csv', index=False)
        print(f"Concordant twins saved to {output_dir}/concordant_twins.csv ({len(results['concordant_twins'])} pairs)")
    
    # Save ambiguous twins
    if results['ambiguous_twins']:
        ambiguous_df = pd.DataFrame(results['ambiguous_twins'])
        ambiguous_df.to_csv(f'{output_dir}/ambiguous_twins.csv', index=False)
        print(f"Ambiguous twins saved to {output_dir}/ambiguous_twins.csv ({len(results['ambiguous_twins'])} pairs)")
    
    # Save all splits
    if results['all_splits']:
        all_splits_df = pd.DataFrame(results['all_splits'])
        all_splits_df.to_csv(f'{output_dir}/all_splits_ratings.csv', index=False)
        print(f"All splits saved to {output_dir}/all_splits_ratings.csv ({len(results['all_splits'])} organoids)")

def plot_twin_analysis(results, output_dir='output'):
    """Create visualizations of twin relationships"""
    import matplotlib.pyplot as plt
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)
    fig.suptitle('Organoid Split Twin Analysis', fontsize=18, fontweight='bold')
    
    summary = results['summary']
    
    # 1. Main plot: Twin rating correlation scatter (LARGER)
    ax = fig.add_subplot(gs[0, :2])  # Span 2 columns
    
    concordant_df = pd.DataFrame(results['concordant_twins']) if results['concordant_twins'] else pd.DataFrame()
    ambiguous_df = pd.DataFrame(results['ambiguous_twins']) if results['ambiguous_twins'] else pd.DataFrame()
    evil_df = pd.DataFrame(results['evil_twins']) if results['evil_twins'] else pd.DataFrame()
    
    # Plot all twin pairs
    if len(concordant_df) > 0:
        ax.scatter(concordant_df['split1_acceptable_rate'], concordant_df['split2_acceptable_rate'],
                  s=150, alpha=0.7, color='#2ecc71', edgecolors='black', linewidth=1.5,
                  label=f'Concordant (n={len(concordant_df)})', zorder=3)
    
    if len(ambiguous_df) > 0:
        ax.scatter(ambiguous_df['split1_acceptable_rate'], ambiguous_df['split2_acceptable_rate'],
                  s=150, alpha=0.7, color='#f39c12', edgecolors='black', linewidth=1.5,
                  label=f'Ambiguous (n={len(ambiguous_df)})', zorder=2)
    
    if len(evil_df) > 0:
        ax.scatter(evil_df['split1_acceptable_rate'], evil_df['split2_acceptable_rate'],
                  s=200, alpha=0.8, color='#e74c3c', edgecolors='black', linewidth=2,
                  marker='X', label=f'Evil Twins (n={len(evil_df)})', zorder=4)
    
    # Add diagonal line
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2.5, label='Perfect correlation', alpha=0.7, zorder=1)
    
    # Quadrants
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.axvline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    
    # Labels for quadrants
    ax.text(0.25, 0.95, 'Split 1: Bad\nSplit 2: Good', ha='center', va='top', 
           fontsize=9, alpha=0.6, style='italic')
    ax.text(0.75, 0.95, 'Both Good', ha='center', va='top',
           fontsize=9, alpha=0.6, style='italic', weight='bold')
    ax.text(0.25, 0.05, 'Both Bad', ha='center', va='bottom',
           fontsize=9, alpha=0.6, style='italic', weight='bold')
    ax.text(0.75, 0.05, 'Split 1: Good\nSplit 2: Bad', ha='center', va='bottom',
           fontsize=9, alpha=0.6, style='italic')
    
    ax.set_xlabel('Split 1 Acceptable Rate', fontsize=13, fontweight='bold')
    ax.set_ylabel('Split 2 Acceptable Rate', fontsize=13, fontweight='bold')
    ax.set_title('Twin Rating Correlation: Do Split Siblings Get Similar Ratings?', 
                fontsize=14, fontweight='bold', pad=15)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc='upper left', fontsize=11, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    # 2. Count summary (top right)
    ax = fig.add_subplot(gs[0, 2])
    
    categories = ['Concordant\n(same, strong)', 'Ambiguous\n(same, weak)', 'Evil Twins\n(opposite)']
    counts = [summary['concordant_twins_count'], summary['ambiguous_twins_count'], summary['evil_twins_count']]
    colors_bar = ['#2ecc71', '#f39c12', '#e74c3c']
    
    bars = ax.bar(range(len(categories)), counts, color=colors_bar, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Add count labels on bars
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{count}\n({count/(sum(counts))*100:.1f}%)',
               ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    ax.set_xticks(range(len(categories)))
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylabel('Number of Twin Pairs', fontsize=11, fontweight='bold')
    ax.set_title('Twin Relationship Counts', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, max(counts) * 1.2 if max(counts) > 0 else 1)
    
    # 3. Agreement strength comparison (bottom left)
    ax = fig.add_subplot(gs[1, 0])
    
    if len(concordant_df) > 0:
        all_strengths_concordant = np.concatenate([
            concordant_df['split1_strength'].values,
            concordant_df['split2_strength'].values
        ])
        ax.hist(all_strengths_concordant, bins=15, alpha=0.7, color='#2ecc71', 
               edgecolor='black', label='Concordant')
    
    if len(ambiguous_df) > 0:
        all_strengths_ambiguous = np.concatenate([
            ambiguous_df['split1_strength'].values,
            ambiguous_df['split2_strength'].values
        ])
        ax.hist(all_strengths_ambiguous, bins=15, alpha=0.7, color='#f39c12',
               edgecolor='black', label='Ambiguous')
    
    ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold (50%)', alpha=0.7)
    ax.set_xlabel('Agreement Strength\n(|Acceptable - Not Acceptable| / Total)', fontsize=10)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Rater Agreement Strength', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 4. Rating distribution for all splits (bottom middle)
    ax = fig.add_subplot(gs[1, 1])
    all_splits_df = pd.DataFrame(results['all_splits'])
    
    if len(all_splits_df) > 0:
        ax.hist(all_splits_df['acceptable_rate'], bins=10, color='#9b59b6', 
               alpha=0.7, edgecolor='black')
        ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='50% threshold')
        
        # Add text showing how many are mostly acceptable
        mostly_acceptable = sum(all_splits_df['acceptable_rate'] > 0.5)
        mostly_not = len(all_splits_df) - mostly_acceptable
        
        ax.text(0.75, ax.get_ylim()[1] * 0.9, 
               f'Mostly Acceptable: {mostly_acceptable}\nMostly Not: {mostly_not}',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
               fontsize=9)
    
    ax.set_xlabel('Acceptable Rate', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Overall Split Quality Distribution', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 5. Summary statistics (bottom right)
    ax = fig.add_subplot(gs[1, 2])
    
    summary_text = f"""
SUMMARY STATISTICS

Total Dy30 with surveys: {summary['total_dy30']}
Total split organoids: {summary['total_splits']}
Wells with multiple splits: {summary['wells_with_multiple_splits']}

Average raters per split: {summary['avg_raters_per_split']:.1f}
Average inter-rater agreement: {summary['avg_inter_rater_agreement']:.1f}%

TWIN PAIRS: {summary['concordant_twins_count'] + summary['ambiguous_twins_count'] + summary['evil_twins_count']}
  • Concordant: {summary['concordant_twins_count']} ({summary['concordant_twins_count']/(summary['concordant_twins_count'] + summary['ambiguous_twins_count'] + summary['evil_twins_count'])*100:.1f}%)
  • Ambiguous: {summary['ambiguous_twins_count']} ({summary['ambiguous_twins_count']/(summary['concordant_twins_count'] + summary['ambiguous_twins_count'] + summary['evil_twins_count'])*100:.1f}%)
  • Evil Twins: {summary['evil_twins_count']} ({summary['evil_twins_count']/(summary['concordant_twins_count'] + summary['ambiguous_twins_count'] + summary['evil_twins_count'])*100 if (summary['concordant_twins_count'] + summary['ambiguous_twins_count'] + summary['evil_twins_count']) > 0 else 0:.1f}%)
    """
    
    ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
           fontsize=10, verticalalignment='top', family='monospace',
           bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.3))
    ax.axis('off')
    
    plt.savefig(f'{output_dir}/twin_analysis_summary.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"\n✓ Visualization saved to {output_dir}/twin_analysis_summary.png")

def print_summary(results):
    """Print summary statistics"""
    summary = results['summary']
    
    print("\n" + "="*60)
    print("ORGANOID SPLIT ANALYSIS SUMMARY")
    print("="*60)
    print(f"\nTotal Dy30 organoids with surveys: {summary['total_dy30']}")
    print(f"Total split organoids: {summary['total_splits']}")
    print(f"Wells with multiple splits: {summary['wells_with_multiple_splits']}")
    print(f"Average raters per split: {summary['avg_raters_per_split']:.1f}")
    print(f"Average inter-rater agreement: {summary['avg_inter_rater_agreement']:.1f}%")
    
    print("\n" + "-"*60)
    print("TWIN RELATIONSHIP BREAKDOWN")
    print("-"*60)
    print(f"Evil Twins (opposite consensus): {summary['evil_twins_count']}")
    print(f"Concordant Twins (same consensus, strong): {summary['concordant_twins_count']}")
    print(f"Ambiguous Twins (same consensus, weak): {summary['ambiguous_twins_count']}")
    
    if results['evil_twins']:
        print("\n" + "-"*60)
        print("EVIL TWIN EXAMPLES (first 10)")
        print("-"*60)
        for i, twin in enumerate(results['evil_twins'][:10], 1):
            print(f"\n{i}. Well: {twin['well']}")
            print(f"   Split {twin['split1_index']}: {twin['split1_consensus']} "
                  f"({twin['split1_acceptable_rate']*100:.0f}% acceptable)")
            print(f"   Split {twin['split2_index']}: {twin['split2_consensus']} "
                  f"({twin['split2_acceptable_rate']*100:.0f}% acceptable)")
            print(f"   Raters: {twin['rater_count']}")

# Main execution
if __name__ == "__main__":
    # CHANGE THIS to your JSON file path
    input_file = ALL_DATA_JSON
    
    print(f"Loading data from {input_file}...")
    data = load_organoid_data(input_file)
    print(f"Loaded {len(data)} total organoid entries")
    
    print("\nAnalyzing splits...")
    results = analyze_splits(data)
    
    print_summary(results)
    
    print("\nSaving results...")
    save_results(results, output_dir='organoid_analysis_output')
    
    print("\nGenerating visualizations...")
    plot_twin_analysis(results, output_dir='organoid_analysis_output')
    print("\nGenerating visualizations...")
    plot_twin_analysis(results, output_dir='organoid_analysis_output')
    plot_area_based_charts(data, results, output_dir='organoid_analysis_output')

    
    print("\n✓ Analysis complete!")

Loading data from /net/projects2/promega/data-analysis/output/all_data.json...
Loaded 5168 total organoid entries

Analyzing splits...
Found 369 Dy30 organoids with survey data
Found 13 wells with multiple splits

ORGANOID SPLIT ANALYSIS SUMMARY

Total Dy30 organoids with surveys: 369
Total split organoids: 27
Wells with multiple splits: 13
Average raters per split: 5.4
Average inter-rater agreement: 85.2%

------------------------------------------------------------
TWIN RELATIONSHIP BREAKDOWN
------------------------------------------------------------
Evil Twins (opposite consensus): 9
Concordant Twins (same consensus, strong): 3
Ambiguous Twins (same consensus, weak): 1

------------------------------------------------------------
EVIL TWIN EXAMPLES (first 10)
------------------------------------------------------------

1. Well: BA2_96_1_Dy30_C1
   Split 1: Acceptable (60% acceptable)
   Split 2: Not Acceptable (0% acceptable)
   Raters: 5

2. Well: BA2_96_1_Dy30_D10
   Split 1: N

In [11]:
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# ==========
# I/O + HELPERS
# ==========

def load_organoid_data(filepath):
    """Load JSON file, handling NaN values."""
    with open(filepath, 'r') as f:
        content = f.read().replace('NaN', 'null')
        data = json.loads(content)
    return data

def normalize_ba(ba_str):
    """Normalize BA string for comparison."""
    return ba_str.replace(' ', '').replace('_', '').lower()

def extract_mask_area(mask_path):
    """Extract area from binary mask (count non-zero pixels)."""
    try:
        if not mask_path or not Path(mask_path).exists():
            return None
        mask = Image.open(mask_path).convert('L')
        mask_array = np.array(mask)
        area_pixels = np.sum(mask_array > 0)
        return area_pixels
    except Exception as e:
        print(f"Error reading mask {mask_path}: {e}")
        return None

def calculate_area_um2(area_pixels, um_per_px_x, um_per_px_y):
    """Convert pixel area to µm²."""
    if area_pixels is None or um_per_px_x is None or um_per_px_y is None:
        return None
    return area_pixels * um_per_px_x * um_per_px_y

# ==========
# SIZE / GROWTH ANALYSIS
# ==========

def build_size_timeline(data):
    """Build timeline of size measurements for all organoids."""
    all_days = ['Dy03', 'Dy06', 'Dy08', 'Dy10', 'Dy13',
                'Dy15', 'Dy17', 'Dy21', 'Dy24', 'Dy28', 'Dy30']
    
    # Group by BA + wellID
    well_groups = defaultdict(lambda: defaultdict(list))
    
    for key, org in data.items():
        ba = org.get('BA', '')
        well = org.get('wellID', '')
        day = org.get('dayID', '')
        
        if not ba or not well or not day:
            continue
        
        ba_norm = normalize_ba(ba)
        well_key = f"{ba_norm}_{well}"
        
        processed = org.get('processed', {})
        mask_path = processed.get('mask_path', '')
        um_per_px_x = processed.get('final_um_per_px_x')
        um_per_px_y = processed.get('final_um_per_px_y')
        main_id = org.get('main_id', key)
        
        # Split info
        is_split = '_split' in main_id
        split_match = re.search(r'_split(\d+)_', main_id)
        split_index = int(split_match.group(1)) if split_match else 0
        
        # Simple majority rating
        rating = None
        if 'survey' in org and 'evaluations' in org['survey']:
            evals = org['survey']['evaluations']
            if evals:
                acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
                total = len(evals)
                rating = 'Acceptable' if acceptable > total/2 else 'Not Acceptable'
        
        area_pixels = extract_mask_area(mask_path)
        area_um2 = calculate_area_um2(area_pixels, um_per_px_x, um_per_px_y)
        
        well_groups[well_key][day].append({
            'main_id': main_id,
            'day': day,
            'day_num': int(day.replace('Dy', '')),
            'is_split': is_split,
            'split_index': split_index,
            'mask_path': mask_path,
            'area_pixels': area_pixels,
            'area_um2': area_um2,
            'um_per_px_x': um_per_px_x,
            'um_per_px_y': um_per_px_y,
            'rating': rating,
            'ba': ba,
            'well': well
        })
    
    return well_groups, all_days

def create_growth_curves(well_groups, all_days, output_dir, max_wells=15):
    """Create growth curve plots for wells that split by Dy30."""
    Path(output_dir).mkdir(exist_ok=True)
    
    split_wells = {}
    nonsplit_wells = {}
    
    for well_key, timeline in well_groups.items():
        if 'Dy30' in timeline:
            has_split = any(org['is_split'] for org in timeline['Dy30'])
            if has_split:
                split_wells[well_key] = timeline
            else:
                nonsplit_wells[well_key] = timeline
    
    print(f"\nFound {len(split_wells)} wells with splits")
    print(f"Found {len(nonsplit_wells)} wells without splits")
    
    fig, axes = plt.subplots(3, 5, figsize=(20, 12))
    axes = axes.flatten()
    fig.suptitle('Growth Curves: Wells That Split by Dy30', fontsize=16, fontweight='bold')
    
    for idx, (well_key, timeline) in enumerate(list(split_wells.items())[:max_wells]):
        ax = axes[idx]
        for day_idx, day in enumerate(all_days):
            if day not in timeline:
                continue
            for org in timeline[day]:
                if org['area_um2'] is None:
                    continue
                if not org['is_split']:
                    color, marker = 'blue', 'o'
                else:
                    color = 'red' if org['split_index'] == 1 else 'orange'
                    marker = 's'
                ax.scatter(day_idx, org['area_um2'], color=color, marker=marker,
                           s=50, alpha=0.7)
        
        ax.set_xticks(range(len(all_days)))
        ax.set_xticklabels(all_days, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Area (µm²)', fontsize=9)
        ax.set_title(well_key.replace('_', ' ')[:20], fontsize=9)
        ax.grid(True, alpha=0.3)
    
    # Hide unused axes
    for idx in range(len(list(split_wells.items())[:max_wells]), len(axes)):
        axes[idx].axis('off')
    
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='blue', label='Presplit'),
        Patch(facecolor='red', label='Split 1'),
        Patch(facecolor='orange', label='Split 2')
    ]
    fig.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    out_path = f"{output_dir}/growth_curves_split_wells.png"
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Growth curves saved to {out_path}")

def compare_split_vs_nonsplit(well_groups, output_dir):
    """Compare size distributions: splits vs non-splits at Dy30."""
    dy30_splits = []
    dy30_nonsplits = []
    
    for well_key, timeline in well_groups.items():
        if 'Dy30' not in timeline:
            continue
        for org in timeline['Dy30']:
            if org['area_um2'] is None:
                continue
            row = {
                'well': well_key,
                'area_um2': org['area_um2'],
                'area_pixels': org['area_pixels'],
                'rating': org['rating'],
                'is_split': org['is_split'],
                'split_index': org['split_index']
            }
            if org['is_split']:
                dy30_splits.append(row)
            else:
                dy30_nonsplits.append(row)
    
    splits_df = pd.DataFrame(dy30_splits)
    nonsplits_df = pd.DataFrame(dy30_nonsplits)
    
    print("\nDy30 Comparison:")
    print(f"  Splits: {len(splits_df)} organoids")
    print(f"  Non-splits: {len(nonsplits_df)} organoids")
    
    if len(splits_df) > 0:
        print(f"  Split mean area: {splits_df['area_um2'].mean():.0f} µm²")
        print(f"  Split median area: {splits_df['area_um2'].median():.0f} µm²")
    if len(nonsplits_df) > 0:
        print(f"  Non-split mean area: {nonsplits_df['area_um2'].mean():.0f} µm²")
        print(f"  Non-split median area: {nonsplits_df['area_um2'].median():.0f} µm²")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # (1) Size hist
    ax = axes[0]
    if len(nonsplits_df) > 0:
        ax.hist(nonsplits_df['area_um2'], bins=30, alpha=0.5,
                label='Non-split', color='blue')
    if len(splits_df) > 0:
        ax.hist(splits_df['area_um2'], bins=30, alpha=0.5,
                label='Split', color='red')
    ax.set_xlabel('Area (µm²)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Size Distribution at Dy30', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # (2) Box plot
    ax = axes[1]
    data_to_plot, labels_to_plot = [], []
    if len(nonsplits_df) > 0:
        data_to_plot.append(nonsplits_df['area_um2'])
        labels_to_plot.append('Non-split')
    if len(splits_df) > 0:
        data_to_plot.append(splits_df['area_um2'])
        labels_to_plot.append('Split')
    if data_to_plot:
        bp = ax.boxplot(data_to_plot, labels=labels_to_plot, patch_artist=True)
        # coloring
        for i, box in enumerate(bp['boxes']):
            box.set_facecolor('blue' if labels_to_plot[i] == 'Non-split' else 'red')
    ax.set_ylabel('Area (µm²)', fontsize=12)
    ax.set_title('Size Comparison', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # (3) Rating breakdown for splits
    ax = axes[2]
    if len(splits_df) > 0 and 'rating' in splits_df.columns:
        rated_splits = splits_df[splits_df['rating'].notna()]
        acceptable = rated_splits[rated_splits['rating'] == 'Acceptable']
        not_acceptable = rated_splits[rated_splits['rating'] == 'Not Acceptable']
        ax.bar(['Acceptable', 'Not Acceptable'],
               [len(acceptable), len(not_acceptable)],
               color=['green', 'red'], alpha=0.6)
        ax.set_ylabel('Count', fontsize=12)
        ax.set_title('Split Ratings at Dy30', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        if len(acceptable) > 0:
            ax.text(0, len(acceptable) + 1,
                    f'Mean: {acceptable["area_um2"].mean():.0f} µm²',
                    ha='center', fontsize=9)
        if len(not_acceptable) > 0:
            ax.text(1, len(not_acceptable) + 1,
                    f'Mean: {not_acceptable["area_um2"].mean():.0f} µm²',
                    ha='center', fontsize=9)
    
    plt.tight_layout()
    out_path = f"{output_dir}/dy30_split_comparison.png"
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Comparison plot saved to {out_path}")
    
    if len(splits_df) > 0:
        splits_df.to_csv(f"{output_dir}/dy30_splits_sizes.csv", index=False)
    if len(nonsplits_df) > 0:
        nonsplits_df.to_csv(f"{output_dir}/dy30_nonsplits_sizes.csv", index=False)

def analyze_growth_rates(well_groups, output_dir):
    """Calculate and compare per-well growth rates."""
    growth_data = []
    
    for well_key, timeline in well_groups.items():
        measurements = []
        for day, orgs in timeline.items():
            for org in orgs:
                if org['area_um2'] is None:
                    continue
                measurements.append({
                    'day': day,
                    'day_num': org['day_num'],
                    'area_um2': org['area_um2'],
                    'is_split': org['is_split'],
                })
        if len(measurements) < 2:
            continue
        measurements.sort(key=lambda x: x['day_num'])
        initial = measurements[0]
        final = measurements[-1]
        days_elapsed = final['day_num'] - initial['day_num']
        if days_elapsed <= 0:
            continue
        growth_rate = (final['area_um2'] - initial['area_um2']) / days_elapsed
        growth_data.append({
            'well': well_key,
            'initial_area': initial['area_um2'],
            'final_area': final['area_um2'],
            'days_elapsed': days_elapsed,
            'growth_rate_um2_per_day': growth_rate,
            'has_split_at_end': final['is_split'],
            'n_timepoints': len(measurements)
        })
    
    growth_df = pd.DataFrame(growth_data)
    if len(growth_df) == 0:
        print("\nNo growth data available.")
        return
    
    print("\nGrowth Rate Analysis:")
    print(f"  Total tracked wells: {len(growth_df)}")
    
    split_growth = growth_df[growth_df['has_split_at_end'] == True]
    nonsplit_growth = growth_df[growth_df['has_split_at_end'] == False]
    
    print(f"\n  Wells that split: {len(split_growth)}")
    if len(split_growth) > 0:
        print(f"    Mean growth rate: {split_growth['growth_rate_um2_per_day'].mean():.1f} µm²/day")
        print(f"    Median growth rate: {split_growth['growth_rate_um2_per_day'].median():.1f} µm²/day")
    
    print(f"\n  Wells that didn't split: {len(nonsplit_growth)}")
    if len(nonsplit_growth) > 0:
        print(f"    Mean growth rate: {nonsplit_growth['growth_rate_um2_per_day'].mean():.1f} µm²/day")
        print(f"    Median growth rate: {nonsplit_growth['growth_rate_um2_per_day'].median():.1f} µm²/day")
    
    growth_df.to_csv(f"{output_dir}/growth_rates.csv", index=False)
    print(f"✓ Growth rates saved to {output_dir}/growth_rates.csv")
    
    fig, ax = plt.subplots(figsize=(10, 6))
    if len(split_growth) > 0:
        ax.hist(split_growth['growth_rate_um2_per_day'], bins=20, alpha=0.5,
                label=f'Split (n={len(split_growth)})', color='red', density=True)
    if len(nonsplit_growth) > 0:
        ax.hist(nonsplit_growth['growth_rate_um2_per_day'], bins=20, alpha=0.5,
                label=f'Non-split (n={len(nonsplit_growth)})', color='blue', density=True)
    ax.set_xlabel('Growth Rate (µm²/day)', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title('Growth Rate Distribution (Normalized)', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = f"{output_dir}/growth_rate_distribution.png"
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Growth rate plot saved to {out_path}")

# ==========
# SPLIT / TWIN ANALYSIS
# ==========

def analyze_splits(data):
    """Analyze organoid splits and inter-rater agreement at Dy30."""
    dy30_organoids = {}
    for key, org in data.items():
        if (org.get('dayID') == 'Dy30' and 
            org.get('survey') and 
            org['survey'].get('evaluations') and 
            len(org['survey']['evaluations']) > 0):
            dy30_organoids[key] = org
    
    print(f"\nFound {len(dy30_organoids)} Dy30 organoids with survey data")
    
    well_groups = defaultdict(list)
    for key, org in dy30_organoids.items():
        main_id = org.get('main_id', key)
        split_match = re.search(r'_split(\d+)_', main_id)
        if split_match:
            split_index = int(split_match.group(1))
            parent_well = main_id[:split_match.start()]
        elif '_nosplit' in main_id:
            parent_well = main_id[:main_id.rfind('_nosplit')]
            split_index = 0
        elif '_stitched' in main_id:
            parent_well = main_id[:main_id.rfind('_stitched')]
            split_index = -1
        else:
            parent_well = main_id
            split_index = 0
        
        evals = org['survey']['evaluations']
        acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
        not_acceptable = sum(1 for e in evals if e.get('evaluation') == 'Not Acceptable')
        total = len(evals)
        
        well_groups[parent_well].append({
            'main_id': main_id,
            'split_index': split_index,
            'evaluations': evals,
            'acceptable': acceptable,
            'not_acceptable': not_acceptable,
            'total': total,
            'acceptable_rate': acceptable / total if total > 0 else 0,
            'consensus': 'Acceptable' if acceptable > not_acceptable else 'Not Acceptable',
            'agreement_strength': abs(acceptable - not_acceptable) / total if total > 0 else 0
        })
    
    split_wells = {well: splits for well, splits in well_groups.items() if len(splits) > 1}
    print(f"Found {len(split_wells)} wells with multiple splits")
    
    evil_twins, concordant_twins, ambiguous_twins = [], [], []
    
    for well, splits in split_wells.items():
        splits.sort(key=lambda x: x['split_index'])
        for i in range(len(splits)):
            for j in range(i + 1, len(splits)):
                s1, s2 = splits[i], splits[j]
                twin_pair = {
                    'well': well,
                    'split1_id': s1['main_id'],
                    'split2_id': s2['main_id'],
                    'split1_index': s1['split_index'],
                    'split2_index': s2['split_index'],
                    'split1_consensus': s1['consensus'],
                    'split2_consensus': s2['consensus'],
                    'split1_acceptable_rate': s1['acceptable_rate'],
                    'split2_acceptable_rate': s2['acceptable_rate'],
                    'split1_strength': s1['agreement_strength'],
                    'split2_strength': s2['agreement_strength'],
                    'rater_count': s1['total']
                }
                if s1['consensus'] != s2['consensus']:
                    twin_pair['relationship'] = 'Evil Twins'
                    evil_twins.append(twin_pair)
                elif s1['agreement_strength'] > 0.5 and s2['agreement_strength'] > 0.5:
                    twin_pair['relationship'] = 'Concordant'
                    concordant_twins.append(twin_pair)
                else:
                    twin_pair['relationship'] = 'Ambiguous'
                    ambiguous_twins.append(twin_pair)
    
    all_split_organoids = [
        org for org in dy30_organoids.values()
        if '_split' in org.get('main_id', '')
    ]
    total_splits = len(all_split_organoids)
    avg_raters = (np.mean([len(org['survey']['evaluations'])
                           for org in all_split_organoids])
                  if all_split_organoids else 0)
    
    agreements = []
    for org in all_split_organoids:
        evals = org['survey']['evaluations']
        acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
        total = len(evals)
        agreement = max(acceptable, total - acceptable) / total if total > 0 else 0
        agreements.append(agreement)
    avg_agreement = np.mean(agreements) if agreements else 0
    
    all_splits_df = []
    for org in all_split_organoids:
        evals = org['survey']['evaluations']
        acceptable = sum(1 for e in evals if e.get('evaluation') == 'Acceptable')
        not_acceptable = sum(1 for e in evals if e.get('evaluation') == 'Not Acceptable')
        all_splits_df.append({
            'main_id': org['main_id'],
            'acceptable': acceptable,
            'not_acceptable': not_acceptable,
            'total': len(evals),
            'acceptable_rate': acceptable / len(evals) if len(evals) > 0 else 0
        })
    
    results = {
        'summary': {
            'total_dy30': len(dy30_organoids),
            'total_splits': total_splits,
            'wells_with_multiple_splits': len(split_wells),
            'avg_raters_per_split': avg_raters,
            'avg_inter_rater_agreement': avg_agreement * 100,
            'evil_twins_count': len(evil_twins),
            'concordant_twins_count': len(concordant_twins),
            'ambiguous_twins_count': len(ambiguous_twins)
        },
        'evil_twins': evil_twins,
        'concordant_twins': concordant_twins,
        'ambiguous_twins': ambiguous_twins,
        'all_splits': all_splits_df
    }
    return results

def save_results(results, output_dir):
    Path(output_dir).mkdir(exist_ok=True)
    summary_df = pd.DataFrame([results['summary']])
    summary_df.to_csv(f'{output_dir}/summary_stats.csv', index=False)
    if results['evil_twins']:
        pd.DataFrame(results['evil_twins']).to_csv(f'{output_dir}/evil_twins.csv', index=False)
    if results['concordant_twins']:
        pd.DataFrame(results['concordant_twins']).to_csv(f'{output_dir}/concordant_twins.csv', index=False)
    if results['ambiguous_twins']:
        pd.DataFrame(results['ambiguous_twins']).to_csv(f'{output_dir}/ambiguous_twins.csv', index=False)
    if results['all_splits']:
        pd.DataFrame(results['all_splits']).to_csv(f'{output_dir}/all_splits_ratings.csv', index=False)

def plot_twin_analysis(results, output_dir):
    Path(output_dir).mkdir(exist_ok=True)
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)
    fig.suptitle('Organoid Split Twin Analysis', fontsize=18, fontweight='bold')
    
    summary = results['summary']
    concordant_df = pd.DataFrame(results['concordant_twins']) if results['concordant_twins'] else pd.DataFrame()
    ambiguous_df = pd.DataFrame(results['ambiguous_twins']) if results['ambiguous_twins'] else pd.DataFrame()
    evil_df = pd.DataFrame(results['evil_twins']) if results['evil_twins'] else pd.DataFrame()
    
    # 1. Scatter
    ax = fig.add_subplot(gs[0, :2])
    if len(concordant_df) > 0:
        ax.scatter(concordant_df['split1_acceptable_rate'], concordant_df['split2_acceptable_rate'],
                   s=150, alpha=0.7, color='#2ecc71', edgecolors='black', linewidth=1.5,
                   label=f'Concordant (n={len(concordant_df)})', zorder=3)
    if len(ambiguous_df) > 0:
        ax.scatter(ambiguous_df['split1_acceptable_rate'], ambiguous_df['split2_acceptable_rate'],
                   s=150, alpha=0.7, color='#f39c12', edgecolors='black', linewidth=1.5,
                   label=f'Ambiguous (n={len(ambiguous_df)})', zorder=2)
    if len(evil_df) > 0:
        ax.scatter(evil_df['split1_acceptable_rate'], evil_df['split2_acceptable_rate'],
                   s=200, alpha=0.8, color='#e74c3c', edgecolors='black', linewidth=2,
                   marker='X', label=f'Evil Twins (n={len(evil_df)})', zorder=4)
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2.5, label='Perfect correlation', alpha=0.7, zorder=1)
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.axvline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.text(0.25, 0.95, 'Split 1: Bad\nSplit 2: Good', ha='center', va='top',
            fontsize=9, alpha=0.6, style='italic')
    ax.text(0.75, 0.95, 'Both Good', ha='center', va='top',
            fontsize=9, alpha=0.6, style='italic', weight='bold')
    ax.text(0.25, 0.05, 'Both Bad', ha='center', va='bottom',
            fontsize=9, alpha=0.6, style='italic', weight='bold')
    ax.text(0.75, 0.05, 'Split 1: Good\nSplit 2: Bad', ha='center', va='bottom',
            fontsize=9, alpha=0.6, style='italic')
    ax.set_xlabel('Split 1 Acceptable Rate', fontsize=13, fontweight='bold')
    ax.set_ylabel('Split 2 Acceptable Rate', fontsize=13, fontweight='bold')
    ax.set_title('Twin Rating Correlation', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc='upper left', fontsize=11, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    # 2. Counts
    ax = fig.add_subplot(gs[0, 2])
    categories = ['Concordant\n(same,strong)', 'Ambiguous\n(same,weak)', 'Evil Twins\n(opposite)']
    counts = [summary['concordant_twins_count'],
              summary['ambiguous_twins_count'],
              summary['evil_twins_count']]
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    bars = ax.bar(range(len(categories)), counts, color=colors,
                  alpha=0.8, edgecolor='black', linewidth=1.5)
    total_pairs = sum(counts) if sum(counts) > 0 else 1
    for bar, count in zip(bars, counts):
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2., h,
                f'{count}\n({count/total_pairs*100:.1f}%)',
                ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax.set_xticks(range(len(categories)))
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylabel('Twin Pairs', fontsize=11, fontweight='bold')
    ax.set_title('Twin Relationship Counts', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, max(counts)*1.2 if max(counts) > 0 else 1)
    
    # 3. Agreement strength
    ax = fig.add_subplot(gs[1, 0])
    if len(concordant_df) > 0:
        strengths_c = np.concatenate([concordant_df['split1_strength'].values,
                                      concordant_df['split2_strength'].values])
        ax.hist(strengths_c, bins=15, alpha=0.7, color='#2ecc71',
                edgecolor='black', label='Concordant')
    if len(ambiguous_df) > 0:
        strengths_a = np.concatenate([ambiguous_df['split1_strength'].values,
                                      ambiguous_df['split2_strength'].values])
        ax.hist(strengths_a, bins=15, alpha=0.7, color='#f39c12',
                edgecolor='black', label='Ambiguous')
    ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold (50%)', alpha=0.7)
    ax.set_xlabel('Agreement Strength', fontsize=10)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Rater Agreement Strength', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 4. Acceptable rate distribution
    ax = fig.add_subplot(gs[1, 1])
    all_splits_df = pd.DataFrame(results['all_splits'])
    if len(all_splits_df) > 0:
        ax.hist(all_splits_df['acceptable_rate'], bins=10, color='#9b59b6',
                alpha=0.7, edgecolor='black')
        ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='50% threshold')
        mostly_acceptable = sum(all_splits_df['acceptable_rate'] > 0.5)
        mostly_not = len(all_splits_df) - mostly_acceptable
        ax.text(0.75, ax.get_ylim()[1]*0.9,
                f'Mostly Acceptable: {mostly_acceptable}\nMostly Not: {mostly_not}',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                fontsize=9)
    ax.set_xlabel('Acceptable Rate', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Overall Split Quality Distribution', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 5. Summary text
    ax = fig.add_subplot(gs[1, 2])
    total_pairs = (summary['concordant_twins_count'] +
                   summary['ambiguous_twins_count'] +
                   summary['evil_twins_count'])
    summary_text = f"""
SUMMARY

Total Dy30 with surveys: {summary['total_dy30']}
Total split organoids:  {summary['total_splits']}
Wells with >1 split:    {summary['wells_with_multiple_splits']}

Avg raters per split:   {summary['avg_raters_per_split']:.1f}
Avg inter-rater agree:  {summary['avg_inter_rater_agreement']:.1f}%

Twin pairs: {total_pairs}
  • Concordant: {summary['concordant_twins_count']}
  • Ambiguous:  {summary['ambiguous_twins_count']}
  • Evil Twins: {summary['evil_twins_count']}
"""
    ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
            fontsize=10, va='top', family='monospace',
            bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.3))
    ax.axis('off')
    
    out_path = f'{output_dir}/twin_analysis_summary.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Twin analysis summary saved to {out_path}")

def build_area_lookup_from_masks(data):
    """Return dict: main_id -> area_um2 (computed from masks)."""
    area_lookup = {}
    for key, org in data.items():
        main_id = org.get('main_id', key)
        processed = org.get('processed', {})
        mask_path = processed.get('mask_path', '')
        umx = processed.get('final_um_per_px_x')
        umy = processed.get('final_um_per_px_y')
        area_pixels = extract_mask_area(mask_path)
        area_um2 = calculate_area_um2(area_pixels, umx, umy)
        area_lookup[main_id] = area_um2
    return area_lookup

def plot_area_based_charts(data, results, output_dir):
    """Area vs rating + twin area asymmetry vs rating asymmetry."""
    Path(output_dir).mkdir(exist_ok=True)
    area_lookup = build_area_lookup_from_masks(data)
    
    all_splits_df = pd.DataFrame(results['all_splits'])
    all_splits_df['area_um2'] = all_splits_df['main_id'].map(area_lookup)
    all_splits_df = all_splits_df.dropna(subset=['area_um2'])
    
    if len(all_splits_df) > 0:
        plt.figure(figsize=(7, 5))
        plt.scatter(all_splits_df['area_um2'], all_splits_df['acceptable_rate'],
                    alpha=0.7, edgecolors='black')
        plt.xlabel('Organoid Area (µm²)', fontsize=12)
        plt.ylabel('Acceptable Rate', fontsize=12)
        plt.title('Organoid Area vs Rating', fontsize=13, fontweight='bold')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        out_path = f'{output_dir}/area_vs_rating.png'
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✓ Area vs rating plot saved to {out_path}")
    else:
        print("No area info available for area vs rating plot")
    
    twin_rows = []
    for rel_name, twin_list in [
        ('Concordant', results['concordant_twins']),
        ('Ambiguous', results['ambiguous_twins']),
        ('Evil Twins', results['evil_twins'])
    ]:
        for twin in twin_list:
            a1 = area_lookup.get(twin['split1_id'])
            a2 = area_lookup.get(twin['split2_id'])
            if a1 is None or a2 is None:
                continue
            rating_diff = twin['split2_acceptable_rate'] - twin['split1_acceptable_rate']
            area_diff_norm = (a2 - a1) / ((a1 + a2) / 2.0) if (a1 + a2) > 0 else 0.0
            twin_rows.append({
                'relationship': rel_name,
                'rating_diff': rating_diff,
                'area_diff_norm': area_diff_norm
            })
    
    if len(twin_rows) > 0:
        df_twins = pd.DataFrame(twin_rows)
        plt.figure(figsize=(7, 5))
        for rel_name, color, marker in [
            ('Concordant', '#2ecc71', 'o'),
            ('Ambiguous', '#f39c12', 's'),
            ('Evil Twins', '#e74c3c', 'X')
        ]:
            sub = df_twins[df_twins['relationship'] == rel_name]
            if len(sub) == 0:
                continue
            plt.scatter(sub['area_diff_norm'], sub['rating_diff'],
                        alpha=0.8, edgecolors='black', s=80,
                        label=f'{rel_name} (n={len(sub)})',
                        marker=marker, linewidth=1.3, color=color)
        plt.axhline(0, color='gray', linestyle='--', alpha=0.7)
        plt.axvline(0, color='gray', linestyle='--', alpha=0.7)
        plt.xlabel('Normalized Area Difference\n((A2 - A1) / mean(A1,A2))', fontsize=11)
        plt.ylabel('Rating Difference\n(split2 - split1 acceptable rate)', fontsize=11)
        plt.title('Twin Asymmetry: Size vs Rating', fontsize=13, fontweight='bold')
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=9)
        plt.tight_layout()
        out_path = f'{output_dir}/twin_area_rating_asymmetry.png'
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✓ Twin area–rating asymmetry plot saved to {out_path}")
    else:
        print("No twin pairs with area info for asymmetry plot")

# ==========
# MAIN
# ==========

if __name__ == "__main__":
    # Either define ALL_DATA_JSON earlier or hardcode your path here:
    ALL_DATA_JSON = "/net/projects2/promega/data-analysis/output/all_data.json"
    input_file = ALL_DATA_JSON
    output_dir = "organoid_analysis_output"
    
    print("="*60)
    print("ORGANOID SIZE, GROWTH & SPLIT ANALYSIS")
    print("="*60)
    
    print(f"\nLoading data from {input_file}...")
    data = load_organoid_data(input_file)
    print(f"Loaded {len(data)} organoid entries")
    
    # Size / growth
    print("\nBuilding size timelines...")
    well_groups, all_days = build_size_timeline(data)
    print(f"Tracked {len(well_groups)} unique wells")
    
    print("\nGenerating growth curves...")
    create_growth_curves(well_groups, all_days, output_dir, max_wells=15)
    
    print("\nComparing splits vs non-splits at Dy30...")
    compare_split_vs_nonsplit(well_groups, output_dir)
    
    print("\nAnalyzing growth rates...")
    analyze_growth_rates(well_groups, output_dir)
    
    # Twin analysis
    print("\nAnalyzing splits (ratings)...")
    results = analyze_splits(data)
    
    print("\nSaving split/twin results...")
    save_results(results, output_dir)
    
    print("\nGenerating twin analysis plots...")
    plot_twin_analysis(results, output_dir)
    plot_area_based_charts(data, results, output_dir)
    
    print("\n" + "="*60)
    print("✓ ANALYSIS COMPLETE")
    print("="*60)
    print(f"Check '{output_dir}/' for PNGs + CSVs.")


ORGANOID SIZE, GROWTH & SPLIT ANALYSIS

Loading data from /net/projects2/promega/data-analysis/output/all_data.json...
Loaded 5168 organoid entries

Building size timelines...
Tracked 475 unique wells

Generating growth curves...

Found 14 wells with splits
Found 442 wells without splits
✓ Growth curves saved to organoid_analysis_output/growth_curves_split_wells.png

Comparing splits vs non-splits at Dy30...

Dy30 Comparison:
  Splits: 28 organoids
  Non-splits: 442 organoids
  Split mean area: 1225537 µm²
  Split median area: 1054112 µm²
  Non-split mean area: 1618392 µm²
  Non-split median area: 1331530 µm²


/tmp/ipykernel_563607/1291793447.py:235: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=labels_to_plot, patch_artist=True)


✓ Comparison plot saved to organoid_analysis_output/dy30_split_comparison.png

Analyzing growth rates...

Growth Rate Analysis:
  Total tracked wells: 475

  Wells that split: 14
    Mean growth rate: 40874.8 µm²/day
    Median growth rate: 30384.8 µm²/day

  Wells that didn't split: 461
    Mean growth rate: 48816.1 µm²/day
    Median growth rate: 39700.2 µm²/day
✓ Growth rates saved to organoid_analysis_output/growth_rates.csv
✓ Growth rate plot saved to organoid_analysis_output/growth_rate_distribution.png

Analyzing splits (ratings)...

Found 369 Dy30 organoids with survey data
Found 13 wells with multiple splits

Saving split/twin results...

Generating twin analysis plots...
✓ Twin analysis summary saved to organoid_analysis_output/twin_analysis_summary.png
✓ Area vs rating plot saved to organoid_analysis_output/area_vs_rating.png
✓ Twin area–rating asymmetry plot saved to organoid_analysis_output/twin_area_rating_asymmetry.png

✓ ANALYSIS COMPLETE
Check 'organoid_analysis_output

In [12]:
def analyze_inversion_controls(data):
    """
    For each organoid, split ratings into INV vs NON-INV
    (based on original_image_ref field inside evaluations)
    and compute rating consistency.
    """

    results = []

    for key, org in data.items():
        if 'survey' not in org or 'evaluations' not in org['survey']:
            continue

        evals = org['survey']['evaluations']
        if not evals:
            continue

        inv = []
        normal = []

        for e in evals:
            ref = e.get("original_image_ref", "")
            is_inv = " INV" in ref

            if is_inv:
                inv.append(e)
            else:
                normal.append(e)

        # Skip organoids without both
        if len(inv) == 0 or len(normal) == 0:
            continue
        
        # Compute acceptable rates
        def rate(lst):
            acc = sum(x['evaluation'] == 'Acceptable' for x in lst)
            tot = len(lst)
            return acc / tot if tot > 0 else None
        
        def agreement(lst):
            acc = sum(x['evaluation'] == 'Acceptable' for x in lst)
            tot = len(lst)
            notacc = tot - acc
            return abs(acc - notacc) / tot if tot > 0 else None

        inv_rate = rate(inv)
        normal_rate = rate(normal)
        inv_agree = agreement(inv)
        normal_agree = agreement(normal)

        results.append({
            "main_id": org.get("main_id", key),
            "inv_count": len(inv),
            "normal_count": len(normal),
            "inv_rate": inv_rate,
            "normal_rate": normal_rate,
            "rate_diff": inv_rate - normal_rate,
            "inv_agreement": inv_agree,
            "normal_agreement": normal_agree,
            "agreement_diff": inv_agree - normal_agree,
        })

    return pd.DataFrame(results)

df_inv = analyze_inversion_controls(data)
print(df_inv)
df_inv.to_csv("organoid_analysis_output/inversion_consistency.csv", index=False)



                               main_id  inv_count  normal_count  inv_rate  \
0    BA1_96_1_Dy30_A8_nosplit_nostitch          5             5       0.2   
1    BA1_96_1_Dy30_B4_nosplit_nostitch          5             5       1.0   
2   BA1_96_1_Dy30_C10_nosplit_nostitch          5             5       1.0   
3   BA1_96_1_Dy30_C11_nosplit_nostitch          5             5       1.0   
4   BA1_96_1_Dy30_D11_nosplit_nostitch          5             5       0.8   
5    BA1_96_1_Dy30_G3_nosplit_nostitch          5             5       0.8   
6    BA1_96_1_Dy30_G9_nosplit_nostitch          5             5       0.8   
7    BA2_96_1_Dy30_A6_nosplit_nostitch          5             5       1.0   
8    BA2_96_1_Dy30_B3_nosplit_nostitch          5             5       0.0   
9   BA2_96_1_Dy30_D11_nosplit_nostitch          5             5       1.0   
10   BA2_96_1_Dy30_D3_nosplit_nostitch          5             5       1.0   
11   BA2_96_1_Dy30_E5_nosplit_nostitch          5             5       1.0   

In [19]:
def plot_inv_consistency(df_inv, output_dir='organoid_analysis_output'):
    import os
    import matplotlib.pyplot as plt
    os.makedirs(output_dir, exist_ok=True)

    # 1) Normal vs INV scatter
    plt.figure(figsize=(6, 6))
    plt.scatter(df_inv['normal_rate'], df_inv['inv_rate'],
                s=120, alpha=0.7, edgecolors='black')
    plt.plot([0, 1], [0, 1], 'r--', label='y = x')
    plt.xlabel('Normal acceptable rate')
    plt.ylabel('Inverted acceptable rate')
    plt.title('Normal vs Inverted Ratings')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{output_dir}/inv_normal_scatter.png', dpi=150, bbox_inches='tight')
    plt.close()

    # 2) Histogram of rating differences
    plt.figure(figsize=(7, 4))
    plt.hist(df_inv['rate_diff'], bins=9, edgecolor='black', alpha=0.7)
    plt.axvline(0, color='red', linestyle='--', label='no change')
    plt.xlabel('inv_rate – normal_rate')
    plt.ylabel('Count')
    plt.title('Shift in Rating Due to Inversion')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{output_dir}/inv_rate_diff_hist.png', dpi=150, bbox_inches='tight')
    plt.close()

    # 3) Histogram of agreement changes
    plt.figure(figsize=(7, 4))
    plt.hist(df_inv['agreement_diff'], bins=9, edgecolor='black', alpha=0.7)
    plt.axvline(0, color='red', linestyle='--', label='no change')
    plt.xlabel('inv_agreement – normal_agreement')
    plt.ylabel('Count')
    plt.title('Change in Rater Agreement (INV – Normal)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{output_dir}/inv_agreement_diff_hist.png', dpi=150, bbox_inches='tight')
    plt.close()

    print(f"Saved inversion plots to {output_dir}")
df_inv_norm = pd.read_csv("organoid_analysis_output/inv_flips.csv")

df_inv = analyze_inversion_controls(data)
df_inv.to_csv('organoid_analysis_output/inversion_consistency.csv', index=False)
plot_inv_consistency(df_inv, 'organoid_analysis_output')


Saved inversion plots to organoid_analysis_output


In [15]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def compute_inv_flips(data):
    """
    For each organoid, compare normal vs INV ratings per rater.
    flipped = 1 if that rater changed their call between normal and INV.
    """
    rows = []

    for key, org in data.items():
        evals = org.get('survey', {}).get('evaluations', [])
        if not evals:
            continue

        inv = [e for e in evals if " INV" in e.get("original_image_ref", "")]
        norm = [e for e in evals if " INV" not in e.get("original_image_ref", "")]

        if not inv or not norm:
            continue

        inv_by_rater = {e['employee']: e['evaluation'] for e in inv}
        norm_by_rater = {e['employee']: e['evaluation'] for e in norm}
        shared = set(inv_by_rater.keys()) & set(norm_by_rater.keys())

        for emp in shared:
            flipped = int(inv_by_rater[emp] != norm_by_rater[emp])
            rows.append({
                "main_id": org.get("main_id", key),
                "employee": emp,
                "normal": norm_by_rater[emp],
                "inv": inv_by_rater[emp],
                "flipped": flipped,
            })

    return pd.DataFrame(rows)


def plot_inv_flip_heatmap(df_flips, output_dir="organoid_analysis_output"):
    """
    Heatmap of flips (1) vs no flips (0) per organoid × rater.
    Rows/columns are sorted by how many flips they have.
    """
    os.makedirs(output_dir, exist_ok=True)

    if df_flips.empty:
        print("No INV/normal pairs with shared raters – skipping flip heatmap.")
        return

    # pivot to organoid × rater matrix
    pivot = df_flips.pivot(index="main_id", columns="employee", values="flipped").fillna(0)

    # sort rows and columns by total flips (descending)
    row_order = pivot.sum(axis=1).sort_values(ascending=False).index
    col_order = pivot.sum(axis=0).sort_values(ascending=False).index
    pivot = pivot.loc[row_order, col_order]

    plt.figure(figsize=(8, 6))
    ax = sns.heatmap(
        pivot,
        cmap="Reds",
        vmin=0, vmax=1,
        linewidths=0.5,
        linecolor="lightgray",
        cbar_kws={"label": "Flip (1 = INV rating differs from normal)"},
    )

    plt.title(
        "Inversion Control – Per-Rater Rating Flips\n"
        "Cells show whether a rater changed their call (Acceptable ↔ Not Acceptable)",
        fontsize=12, pad=18, fontweight="bold"
    )
    ax.set_xlabel("Employee")
    ax.set_ylabel("Organoid (main_id)")

    plt.xticks(rotation=45, ha="right")
    plt.yticks(fontsize=8)
    plt.tight_layout()
    out_path = os.path.join(output_dir, "inv_flip_heatmap.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"✓ Saved inversion flip heatmap to {out_path}")

df_flips = compute_inv_flips(data)
df_flips.to_csv("organoid_analysis_output/inv_flips.csv", index=False)
plot_inv_flip_heatmap(df_flips, "organoid_analysis_output")



✓ Saved inversion flip heatmap to organoid_analysis_output/inv_flip_heatmap.png


In [21]:
def compute_flip_summary(df_flips, df_inv_norm):
    """
    df_flips: output of compute_inv_flips()   (per-rater flips)
    df_inv_norm: your 28-row table with inv_rate + normal_rate
    """
    # 1. Total flips across all raters
    total_flips = df_flips['flipped'].sum()

    # 2. Organisms where consensus changed
    # normal_rate > 0.5 → acceptable consensus
    df_inv_norm['normal_consensus'] = (df_inv_norm['normal_rate'] > 0.5).astype(int)
    df_inv_norm['inv_consensus'] = (df_inv_norm['inv_rate'] > 0.5).astype(int)

    df_inv_norm['consensus_changed'] = (
        df_inv_norm['normal_consensus'] != df_inv_norm['inv_consensus']
    ).astype(int)

    flips_affecting_consensus = df_inv_norm['consensus_changed'].sum()

    return total_flips, flips_affecting_consensus

def plot_flip_summary(df_flips, df_inv_norm, output_dir="organoid_analysis_output"):
    import matplotlib.pyplot as plt
    import os
    os.makedirs(output_dir, exist_ok=True)

    # Compute values
    total_flips = df_flips['flipped'].sum()

    df_inv_norm['normal_consensus'] = (df_inv_norm['normal_rate'] > 0.5).astype(int)
    df_inv_norm['inv_consensus'] = (df_inv_norm['inv_rate'] > 0.5).astype(int)
    consensus_changed = (
        df_inv_norm['normal_consensus'] != df_inv_norm['inv_consensus']
    ).sum()

    labels = ["Total flips", "Consensus-changing flips"]
    values = [total_flips, consensus_changed]
    colors = ["#c0392b", "#2980b9"]

    plt.figure(figsize=(6,5))
    bars = plt.bar(labels, values, color=colors)

    # Add value labels
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, val + 0.3, str(val),
                 ha="center", fontsize=12, fontweight="bold")

    plt.ylabel("Count")
    plt.title("Impact of Image Inversion on Ratings", fontsize=14, fontweight="bold")
    plt.tight_layout()

    out_path = f"{output_dir}/flip_summary_bar.png"
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"✓ Saved flip summary bar plot → {out_path}")
def build_flip_summary_table(df_flips, df_inv_norm):
    # Total flips across all rater–organoid pairs
    total_flips = df_flips['flipped'].sum()

    # Number of organoids with ≥1 flip
    orgs_with_flips = df_flips.groupby('main_id')['flipped'].max().sum()

    # Build consensus change flags
    df_inv_norm['normal_consensus'] = (df_inv_norm['normal_rate'] > 0.5).astype(int)
    df_inv_norm['inv_consensus'] = (df_inv_norm['inv_rate'] > 0.5).astype(int)
    df_inv_norm['consensus_changed'] = (
        df_inv_norm['normal_consensus'] != df_inv_norm['inv_consensus']
    ).astype(int)

    consensus_changed = df_inv_norm['consensus_changed'].sum()

    # Percent of flips that changed consensus
    pct_effective = (consensus_changed / orgs_with_flips * 100) if orgs_with_flips > 0 else 0

    summary_df = pd.DataFrame({
        "Metric": [
            "Total rating flips (rater × organoid)",
            "Organoids with ≥1 flip",
            "Organoids with consensus changed",
            "% of flipping organoids where consensus changed"
        ],
        "Value": [
            total_flips,
            orgs_with_flips,
            consensus_changed,
            f"{pct_effective:.1f}%"
        ]
    })

    return summary_df
df_inv_norm = pd.read_csv("organoid_analysis_output/inversion_consistency.csv")
# Build summary table
summary_df = build_flip_summary_table(df_flips, df_inv_norm)

# Print summary table
print(summary_df.to_string(index=False))

# Plot bar chart
plot_flip_summary(df_flips, df_inv_norm, output_dir="organoid_analysis_output")


                                         Metric Value
          Total rating flips (rater × organoid)    16
                         Organoids with ≥1 flip    12
               Organoids with consensus changed     3
% of flipping organoids where consensus changed 25.0%
✓ Saved flip summary bar plot → organoid_analysis_output/flip_summary_bar.png
